In [33]:
#races
import pandas as pd
import requests
from datetime import datetime

# Get the current year and generate a range of the last 5 years
current_year = datetime.now().year
years = range(current_year - 5, current_year + 1)

# Dictionary to collect data
races = {
    'season': [],
    'round': [],
    'circuit_id': [],
    'lat': [],
    'long': [],
    'country': [],
    'date': [],
    'url': []
}

# Loop through each year
for year in years:
    url = f'https://api.jolpi.ca/ergast/f1/{year}/races.json'
    try:
        r = requests.get(url)
        r.raise_for_status()
        if r.headers.get("Content-Type", "").startswith("application/json") and r.text.strip():
            data = r.json()
        else:
            print(f"[{year}] Invalid or empty JSON response.")
            continue
    except Exception as e:
        print(f"[{year}] Error during request or JSON parsing:", e)
        continue

    # Parse races if data was valid
    #for item in data.get('MRData', {}).get('RaceTable', {}).get('Races', []):
    for item in data.get('MRData', {}).get('RaceTable', {}).get('Races', []):
        races['season'].append(int(item.get('season', None)))
    
    # Directly handle None for 'round' key in one line
        races['round'].append(int(item.get('round', 0)) if item.get('round') is not None else 0)
    
        races['circuit_id'].append(item.get('Circuit', {}).get('circuitId', None))
        races['lat'].append(float(item.get('Circuit', {}).get('Location', {}).get('lat', 0.0)))
        races['long'].append(float(item.get('Circuit', {}).get('Location', {}).get('long', 0.0)))
        races['country'].append(item.get('Circuit', {}).get('Location', {}).get('country', None))
        races['date'].append(item.get('date', None))
        races['url'].append(item.get('url', None))

# Convert to DataFrame
races_df = pd.DataFrame(races)
print(races_df.tail(40))

     season  round     circuit_id       lat       long      country  \
97     2025      8         monaco  43.73470    7.42056       Monaco   
98     2025      9      catalunya  41.57000    2.26111        Spain   
99     2025     10     villeneuve  45.50000  -73.52280       Canada   
100    2025     11  red_bull_ring  47.21970   14.76470      Austria   
101    2025     12    silverstone  52.07860   -1.01694           UK   
102    2025     13            spa  50.43720    5.97139      Belgium   
103    2025     14    hungaroring  47.57890   19.24860      Hungary   
104    2025     15      zandvoort  52.38880    4.54092  Netherlands   
105    2025     16          monza  45.61560    9.28111        Italy   
106    2025     17           baku  40.37250   49.85330   Azerbaijan   
107    2025     18     marina_bay   1.29140  103.86400    Singapore   
108    2025     19       americas  30.13280  -97.64110          USA   
109    2025     20      rodriguez  19.40420  -99.09070       Mexico   
110   

In [34]:
#results_improved
import requests
import pandas as pd
import time
import random


SESSION = requests.Session()

HEADERS = {
    "User-Agent": "Mozilla/5.0 (F1 data analysis)"
}


def fetch_season_results(season, max_retries=6):

    all_rows = []
    limit = 100
    offset = 0
    total = None

    while True:

        url = f"https://api.jolpi.ca/ergast/f1/{season}/results"
        params = {
            "limit": limit,
            "offset": offset
        }

        # -----------------------------
        # Request with retry/backoff
        # -----------------------------
        for attempt in range(max_retries):

            try:
                response = SESSION.get(
                    url,
                    params=params,
                    headers=HEADERS,
                    timeout=20
                )
            except requests.RequestException as e:
                wait = 2 ** attempt
                print(
                    f"Request error for {season}, "
                    f"offset {offset}: {e}"
                )
                print(f"Retrying in {wait}s...")
                time.sleep(wait)
                continue

            if response.status_code == 200:
                break

            elif response.status_code == 429:

                # Respect Retry-After if supplied
                retry_after = response.headers.get("Retry-After")

                if retry_after:
                    try:
                        wait = float(retry_after)
                    except ValueError:
                        wait = 2 ** attempt
                else:
                    wait = 2 ** attempt

                # Add a little randomness
                wait += random.uniform(0.5, 1.5)

                print(
                    f"429 rate limit: {season}, "
                    f"offset {offset}"
                )
                print(f"Waiting {wait:.1f}s before retry...")

                time.sleep(wait)

            else:
                print(
                    f"Failed to fetch {season}, "
                    f"status: {response.status_code}"
                )
                return all_rows

        else:
            print(
                f"Giving up on {season}, "
                f"offset {offset} after {max_retries} retries."
            )
            return all_rows

        # -----------------------------
        # Parse JSON
        # -----------------------------

        try:
            data = response.json().get("MRData", {})
        except ValueError:
            print(
                f"Invalid JSON response for "
                f"{season}, offset {offset}"
            )
            return all_rows

        races = data.get("RaceTable", {}).get("Races", [])

        if total is None:
            total = int(data.get("total", 0))
            print(
                f"{season}: {total} result rows available"
            )

        if not races:
            break

        # -----------------------------
        # Extract results
        # -----------------------------

        for race in races:

            round_num = int(race["round"])
            race_name = race.get("raceName")
            circuit = race["Circuit"]["circuitName"]
            date = race.get("date")

            for result in race.get("Results", []):

                driver = result["Driver"]
                constructor = result["Constructor"]

                position = result.get("position")

                try:
                    position = int(position)
                except (TypeError, ValueError):
                    position = None

                all_rows.append({
                    "season": int(season),
                    "round": round_num,
                    "date": date,
                    "race_name": race_name,
                    "circuit": circuit,

                    "driver": driver["driverId"],
                    "driver_name":
                        f"{driver['givenName']} "
                        f"{driver['familyName']}",

                    "constructor":
                        constructor["constructorId"],
                    "constructor_name":
                        constructor["name"],

                    "grid": int(result.get("grid", 0)),
                    "position": position,
                    "status": result.get("status"),
                    "points": float(result.get("points", 0)),
                })

        offset += limit

        if total and offset >= total:
            break

        # -----------------------------
        # IMPORTANT:
        # don't hammer API
        # -----------------------------
        time.sleep(random.uniform(1.0, 2.0))

    print(
        f"Completed {season}: "
        f"{len(all_rows)} driver-race results"
    )

    return all_rows


# ============================================
# Fetch seasons
# ============================================

all_seasons = []

for year in range(2021, 2027):

    print("\n" + "=" * 60)
    print(f"Fetching data for {year}...")
    print("=" * 60)

    season_rows = fetch_season_results(year)

    all_seasons.extend(season_rows)

    # Extra pause between seasons
    time.sleep(3)


# ============================================
# DataFrame
# ============================================

results_df = pd.DataFrame(all_seasons)

if not results_df.empty:

    results_df.sort_values(
        by=["season", "round", "position"],
        inplace=True
    )

    results_df.reset_index(
        drop=True,
        inplace=True
    )

print("\nFinal shape:", results_df.shape)
print(results_df.tail(20))


Fetching data for 2021...
2021: 440 result rows available
Completed 2021: 440 driver-race results

Fetching data for 2022...
2022: 440 result rows available
Completed 2022: 440 driver-race results

Fetching data for 2023...
2023: 440 result rows available
Completed 2023: 440 driver-race results

Fetching data for 2024...
2024: 479 result rows available
Completed 2024: 479 driver-race results

Fetching data for 2025...
2025: 479 result rows available
Completed 2025: 479 driver-race results

Fetching data for 2026...
2026: 264 result rows available
Completed 2026: 264 driver-race results

Final shape: (2542, 13)
      season  round        date         race_name                 circuit  \
2522    2026     12  2026-08-23  Dutch Grand Prix  Circuit Park Zandvoort   
2523    2026     12  2026-08-23  Dutch Grand Prix  Circuit Park Zandvoort   
2524    2026     12  2026-08-23  Dutch Grand Prix  Circuit Park Zandvoort   
2525    2026     12  2026-08-23  Dutch Grand Prix  Circuit Park Zandvoort

In [35]:
# driverstandings - robust version

import pandas as pd
import requests
import time
import random
import os
from datetime import datetime


# ============================================================
# Configuration
# ============================================================

current_year = datetime.now().year

# Last 6 seasons, including current season
years = range(current_year - 5, current_year + 1)

CACHE_DIR = "f1_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

SESSION = requests.Session()

SESSION.headers.update({
    "User-Agent": "Mozilla/5.0 F1 data analysis"
})


# ============================================================
# Robust GET function
# ============================================================

def get_json(url, max_retries=7):

    for attempt in range(max_retries):

        try:
            r = SESSION.get(
                url,
                timeout=20
            )

            # -------------------------
            # Success
            # -------------------------
            if r.status_code == 200:
                return r.json()

            # -------------------------
            # Rate limited
            # -------------------------
            if r.status_code == 429:

                retry_after = r.headers.get("Retry-After")

                if retry_after:
                    try:
                        wait = float(retry_after)
                    except ValueError:
                        wait = 2 ** attempt
                else:
                    wait = 2 ** attempt

                # Small random component
                wait += random.uniform(1, 3)

                print(
                    f"429 rate limit. "
                    f"Retry {attempt + 1}/{max_retries} "
                    f"in {wait:.1f}s..."
                )

                time.sleep(wait)
                continue

            # -------------------------
            # Other HTTP error
            # -------------------------
            print(
                f"HTTP {r.status_code}: {url}"
            )

            return None

        except requests.RequestException as e:

            wait = 2 ** attempt + random.uniform(1, 3)

            print(
                f"Request error: {e}"
            )
            print(
                f"Retry {attempt + 1}/{max_retries} "
                f"in {wait:.1f}s..."
            )

            time.sleep(wait)

    print(f"Giving up: {url}")

    return None


# ============================================================
# 1. Get races
# ============================================================

races_cache = os.path.join(
    CACHE_DIR,
    "races.csv"
)

if os.path.exists(races_cache):

    print("Loading races from cache...")

    races = pd.read_csv(races_cache)

else:

    print("Downloading race calendar...")

    race_rows = []

    for year in years:

        print(f"Fetching races for {year}...")

        url = (
            f"https://api.jolpi.ca/ergast/f1/"
            f"{year}/races.json"
        )

        races_data = get_json(url)

        if races_data is None:
            continue

        race_list = (
            races_data
            .get("MRData", {})
            .get("RaceTable", {})
            .get("Races", [])
        )

        for race in race_list:

            race_rows.append({
                "season": int(race["season"]),
                "round": int(race["round"])
            })

        # Don't hammer API
        time.sleep(random.uniform(2, 4))

    races = pd.DataFrame(race_rows)

    races.to_csv(
        races_cache,
        index=False
    )


print()
print("Races:")
print(races.groupby("season").size())


# ============================================================
# 2. Build season/round list
# ============================================================

rounds = []

for season in sorted(races["season"].unique()):

    round_list = sorted(
        races.loc[
            races["season"] == season,
            "round"
        ].tolist()
    )

    rounds.append(
        [int(season), round_list]
    )


# ============================================================
# 3. Download driver standings
# ============================================================

standing_rows = []


for season, round_list in rounds:

    print()
    print("=" * 60)
    print(f"SEASON {season}")
    print("=" * 60)

    for race_round in round_list:

        cache_file = os.path.join(
            CACHE_DIR,
            f"driverstandings_{season}_{race_round}.csv"
        )

        # -----------------------------------------------
        # Load cached round if already downloaded
        # -----------------------------------------------

        if os.path.exists(cache_file):

            df_cached = pd.read_csv(cache_file)

            standing_rows.extend(
                df_cached.to_dict("records")
            )

            print(
                f"{season} R{race_round}: cached "
                f"({len(df_cached)} drivers)"
            )

            continue


        # -----------------------------------------------
        # API request
        # -----------------------------------------------

        url = (
            f"https://api.jolpi.ca/ergast/f1/"
            f"{season}/{race_round}/"
            f"driverstandings.json"
        )

        print(
            f"{season} R{race_round}: downloading..."
        )

        json_data = get_json(url)

        if json_data is None:
            print(
                f"{season} R{race_round}: FAILED"
            )
            continue


        # -----------------------------------------------
        # Extract standings
        # -----------------------------------------------

        standings = (
            json_data
            .get("MRData", {})
            .get("StandingsTable", {})
            .get("StandingsLists", [])
        )

        if not standings:
            print(
                f"{season} R{race_round}: "
                f"no standings"
            )
            continue


        rows = []

        for item in standings[0].get(
            "DriverStandings",
            []
        ):

            driver = item.get("Driver", {})

            try:
                points = float(
                    item.get("points", 0)
                )
            except:
                points = None

            try:
                wins = int(
                    item.get("wins", 0)
                )
            except:
                wins = None

            try:
                position = int(
                    item.get("position")
                )
            except:
                position = None


            rows.append({

                "season": int(season),

                "round": int(race_round),

                "driver": driver.get(
                    "driverId"
                ),

                "driver_points": points,

                "driver_wins": wins,

                "driver_standings_pos":
                    position
            })


        # -----------------------------------------------
        # Cache this round
        # -----------------------------------------------

        df_round = pd.DataFrame(rows)

        df_round.to_csv(
            cache_file,
            index=False
        )

        standing_rows.extend(rows)

        print(
            f"{season} R{race_round}: "
            f"{len(rows)} drivers"
        )

        # -----------------------------------------------
        # IMPORTANT:
        # slow down requests
        # -----------------------------------------------

        time.sleep(
            random.uniform(3, 6)
        )


# ============================================================
# 4. Final DataFrame
# ============================================================

driver_standings_df = pd.DataFrame(
    standing_rows
)

driver_standings_df.sort_values(
    by=[
        "season",
        "round",
        "driver_standings_pos"
    ],
    inplace=True
)

driver_standings_df.reset_index(
    drop=True,
    inplace=True
)


# ============================================================
# 5. Diagnostics
# ============================================================

print()
print("=" * 60)
print("DRIVER STANDINGS")
print("=" * 60)

print(
    "Shape:",
    driver_standings_df.shape
)

print()

print(
    driver_standings_df.head(20)
)

print()

print(
    driver_standings_df
    .groupby("season")
    .size()
)

Loading races from cache...

Races:
season
2021    22
2022    22
2023    22
2024    24
2025    24
2026    23
dtype: int64

SEASON 2021
2021 R1: cached (20 drivers)
2021 R2: cached (20 drivers)
2021 R3: cached (20 drivers)
2021 R4: cached (20 drivers)
2021 R5: cached (20 drivers)
2021 R6: cached (20 drivers)
2021 R7: cached (20 drivers)
2021 R8: cached (20 drivers)
2021 R9: cached (20 drivers)
2021 R10: cached (20 drivers)
2021 R11: cached (20 drivers)
2021 R12: cached (20 drivers)
2021 R13: cached (21 drivers)
2021 R14: cached (21 drivers)
2021 R15: cached (21 drivers)
2021 R16: cached (21 drivers)
2021 R17: cached (21 drivers)
2021 R18: cached (21 drivers)
2021 R19: cached (21 drivers)
2021 R20: cached (21 drivers)
2021 R21: cached (21 drivers)
2021 R22: cached (21 drivers)

SEASON 2022
2022 R1: cached (20 drivers)
2022 R2: cached (20 drivers)
2022 R3: cached (21 drivers)
2022 R4: cached (21 drivers)
2022 R5: cached (21 drivers)
2022 R6: cached (21 drivers)
2022 R7: cached (21 drivers

In [36]:
print(driver_standings_df.tail(40))

      season  round          driver  driver_points  driver_wins  \
2644    2026     12         piastri          104.0            0   
2645    2026     12          hadjar           68.0            0   
2646    2026     12          lawson           49.0            0   
2647    2026     12           gasly           44.0            0   
2648    2026     12  arvid_lindblad           23.0            0   
2649    2026     12       colapinto           19.0            0   
2650    2026     12         bearman           18.0            0   
2651    2026     12       bortoleto           10.0            0   
2652    2026     12      hulkenberg            6.0            0   
2653    2026     12           sainz            6.0            0   
2654    2026     12           albon            5.0            0   
2655    2026     12            ocon            3.0            0   
2656    2026     12          alonso            3.0            0   
2657    2026     12         tsunoda            0.0            

In [37]:
# ============================================================
# CONSTRUCTOR STANDINGS - ROBUST + CACHED
# ============================================================

import pandas as pd
import requests
import time
import random
import os
from datetime import datetime


# ============================================================
# Configuration
# ============================================================

current_year = datetime.now().year

# 2021 -> current season
years = range(current_year - 5, current_year + 1)

CACHE_DIR = "f1_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

SESSION = requests.Session()

SESSION.headers.update({
    "User-Agent": "Mozilla/5.0 F1 data analysis"
})


# ============================================================
# Robust API request with exponential backoff
# ============================================================

def get_json(url, max_retries=7):

    for attempt in range(max_retries):

        try:

            r = SESSION.get(
                url,
                timeout=20
            )

            # -----------------------------
            # Success
            # -----------------------------

            if r.status_code == 200:
                return r.json()


            # -----------------------------
            # Rate limited
            # -----------------------------

            if r.status_code == 429:

                retry_after = r.headers.get(
                    "Retry-After"
                )

                if retry_after:

                    try:
                        wait = float(retry_after)

                    except ValueError:
                        wait = 2 ** attempt

                else:
                    wait = 2 ** attempt


                wait += random.uniform(1, 3)

                print(
                    f"429 rate limit "
                    f"(attempt {attempt + 1}/{max_retries})"
                )

                print(
                    f"Waiting {wait:.1f}s..."
                )

                time.sleep(wait)

                continue


            # -----------------------------
            # Other HTTP errors
            # -----------------------------

            print(
                f"HTTP {r.status_code}: {url}"
            )

            return None


        except requests.RequestException as e:

            wait = (
                2 ** attempt
                + random.uniform(1, 3)
            )

            print(
                f"Request error: {e}"
            )

            print(
                f"Retrying in {wait:.1f}s..."
            )

            time.sleep(wait)


    print(
        f"Giving up after {max_retries} attempts:"
    )

    print(url)

    return None


# ============================================================
# Step 1: Get race calendar
# ============================================================

races_cache = os.path.join(
    CACHE_DIR,
    "races.csv"
)


if os.path.exists(races_cache):

    print("Loading race calendar from cache...")

    races = pd.read_csv(
        races_cache
    )

else:

    print("Downloading race calendar...")

    race_rows = []


    for year in years:

        print(
            f"Fetching races for {year}..."
        )

        url = (
            f"https://api.jolpi.ca/ergast/f1/"
            f"{year}/races.json"
        )

        data = get_json(url)

        if data is None:
            continue


        race_list = (
            data
            .get("MRData", {})
            .get("RaceTable", {})
            .get("Races", [])
        )


        for race in race_list:

            race_rows.append({

                "season":
                    int(race["season"]),

                "round":
                    int(race["round"])
            })


        # Slow down
        time.sleep(
            random.uniform(2, 4)
        )


    races = pd.DataFrame(
        race_rows
    )


    races.to_csv(
        races_cache,
        index=False
    )


print()
print("Race counts:")
print(
    races.groupby("season").size()
)


# ============================================================
# Step 2: Download constructor standings
# ============================================================

standing_rows = []


for season in sorted(
    races["season"].unique()
):

    season_rounds = sorted(
        races.loc[
            races["season"] == season,
            "round"
        ].tolist()
    )


    print()
    print("=" * 60)
    print(f"SEASON {season}")
    print("=" * 60)


    for race_round in season_rounds:

        # -----------------------------------------
        # Cache one file per season/round
        # -----------------------------------------

        cache_file = os.path.join(
            CACHE_DIR,
            f"constructorstandings_{season}_{race_round}.csv"
        )


        # -----------------------------------------
        # Already downloaded?
        # -----------------------------------------

        if os.path.exists(cache_file):

            cached = pd.read_csv(
                cache_file
            )

            standing_rows.extend(
                cached.to_dict("records")
            )

            print(
                f"{season} R{race_round}: "
                f"cached ({len(cached)} constructors)"
            )

            continue


        # -----------------------------------------
        # API request
        # -----------------------------------------

        url = (
            f"https://api.jolpi.ca/ergast/f1/"
            f"{season}/{race_round}/"
            f"constructorstandings.json"
        )


        print(
            f"{season} R{race_round}: "
            f"downloading..."
        )


        json_data = get_json(url)


        if json_data is None:

            print(
                f"{season} R{race_round}: FAILED"
            )

            continue


        # -----------------------------------------
        # Extract standings
        # -----------------------------------------

        standings = (
            json_data
            .get("MRData", {})
            .get("StandingsTable", {})
            .get("StandingsLists", [])
        )


        if not standings:

            print(
                f"{season} R{race_round}: "
                f"no standings"
            )

            continue


        rows = []


        for item in standings[0].get(
            "ConstructorStandings",
            []
        ):

            constructor = item.get(
                "Constructor",
                {}
            )


            # Points

            try:

                points = float(
                    item.get(
                        "points",
                        0
                    )
                )

            except:

                points = None


            # Wins

            try:

                wins = int(
                    item.get(
                        "wins",
                        0
                    )
                )

            except:

                wins = None


            # Championship position

            try:

                position = int(
                    item.get(
                        "position"
                    )
                )

            except:

                position = None


            rows.append({

                "season":
                    int(season),

                "round":
                    int(race_round),

                "constructor":
                    constructor.get(
                        "constructorId"
                    ),

                "constructor_points":
                    points,

                "constructor_wins":
                    wins,

                "constructor_standings_pos":
                    position
            })


        # -----------------------------------------
        # Save this round immediately
        # -----------------------------------------

        df_round = pd.DataFrame(
            rows
        )


        df_round.to_csv(
            cache_file,
            index=False
        )


        standing_rows.extend(
            rows
        )


        print(
            f"{season} R{race_round}: "
            f"{len(rows)} constructors"
        )


        # -----------------------------------------
        # Don't hammer API
        # -----------------------------------------

        time.sleep(
            random.uniform(3, 6)
        )


# ============================================================
# Step 3: Final DataFrame
# ============================================================

constructor_standings_df = pd.DataFrame(
    standing_rows
)


constructor_standings_df.sort_values(
    by=[
        "season",
        "round",
        "constructor_standings_pos"
    ],
    inplace=True
)


constructor_standings_df.reset_index(
    drop=True,
    inplace=True
)


# ============================================================
# Step 4: Create PREVIOUS-ROUND features
# ============================================================

constructor_standings_df = (
    constructor_standings_df
    .sort_values(
        ["constructor", "season", "round"]
    )
)


# Previous round points
constructor_standings_df[
    "constructor_points_before"
] = (
    constructor_standings_df
    .groupby(
        ["season", "constructor"]
    )["constructor_points"]
    .shift(1)
)


# Previous round wins
constructor_standings_df[
    "constructor_wins_before"
] = (
    constructor_standings_df
    .groupby(
        ["season", "constructor"]
    )["constructor_wins"]
    .shift(1)
)


# Previous championship position
constructor_standings_df[
    "constructor_standings_pos_before"
] = (
    constructor_standings_df
    .groupby(
        ["season", "constructor"]
    )["constructor_standings_pos"]
    .shift(1)
)


# ============================================================
# Step 5: Fill missing previous-round values
# ============================================================

constructor_standings_df[
    "constructor_points_before"
] = (
    constructor_standings_df[
        "constructor_points_before"
    ]
    .fillna(0)
)


constructor_standings_df[
    "constructor_wins_before"
] = (
    constructor_standings_df[
        "constructor_wins_before"
    ]
    .fillna(0)
)


# For championship position:
# first race -> no previous championship position
# using 0 is consistent with your original lookup()
constructor_standings_df[
    "constructor_standings_pos_before"
] = (
    constructor_standings_df[
        "constructor_standings_pos_before"
    ]
    .fillna(0)
)


# ============================================================
# Step 6: Final diagnostics
# ============================================================

print()
print("=" * 60)
print("CONSTRUCTOR STANDINGS COMPLETE")
print("=" * 60)

print(
    "Shape:",
    constructor_standings_df.shape
)

print()

print(
    constructor_standings_df.head(20)
)

print()

print(
    constructor_standings_df[
        [
            "season",
            "round",
            "constructor",
            "constructor_points",
            "constructor_points_before",
            "constructor_wins",
            "constructor_wins_before",
            "constructor_standings_pos",
            "constructor_standings_pos_before"
        ]
    ].head(20)
)

Loading race calendar from cache...

Race counts:
season
2021    22
2022    22
2023    22
2024    24
2025    24
2026    23
dtype: int64

SEASON 2021
2021 R1: cached (10 constructors)
2021 R2: cached (10 constructors)
2021 R3: cached (10 constructors)
2021 R4: cached (10 constructors)
2021 R5: cached (10 constructors)
2021 R6: cached (10 constructors)
2021 R7: cached (10 constructors)
2021 R8: cached (10 constructors)
2021 R9: cached (10 constructors)
2021 R10: cached (10 constructors)
2021 R11: cached (10 constructors)
2021 R12: cached (10 constructors)
2021 R13: cached (10 constructors)
2021 R14: cached (10 constructors)
2021 R15: cached (10 constructors)
2021 R16: cached (10 constructors)
2021 R17: cached (10 constructors)
2021 R18: cached (10 constructors)
2021 R19: cached (10 constructors)
2021 R20: cached (10 constructors)
2021 R21: cached (10 constructors)
2021 R22: cached (10 constructors)

SEASON 2022
2022 R1: cached (10 constructors)
2022 R2: cached (10 constructors)
2022 R3: 

In [38]:
print(
    constructor_standings_df.tail(20)
)

      season  round constructor  constructor_points  constructor_wins  \
1074    2025     18    williams               102.0                 0   
1084    2025     19    williams               111.0                 0   
1094    2025     20    williams               111.0                 0   
1104    2025     21    williams               111.0                 0   
1114    2025     22    williams               121.0                 0   
1124    2025     23    williams               137.0                 0   
1134    2025     24    williams               137.0                 0   
1148    2026      1    williams                 0.0                 0   
1159    2026      2    williams                 2.0                 0   
1170    2026      3    williams                 2.0                 0   
1180    2026      4    williams                 5.0                 0   
1191    2026      5    williams                 7.0                 0   
1202    2026      6    williams                11.0

In [39]:
# ============================================================
# QUALIFYING RESULTS - ROBUST + CACHED
# ============================================================

import requests
import pandas as pd
import time
import random
import os


CACHE_DIR = "f1_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

SESSION = requests.Session()

SESSION.headers.update({
    "User-Agent": "Mozilla/5.0 F1 data analysis"
})


# ============================================================
# Robust API request
# ============================================================

def get_json(url, max_retries=7):

    for attempt in range(max_retries):

        try:

            r = SESSION.get(
                url,
                timeout=20
            )

            if r.status_code == 200:
                return r.json()


            if r.status_code == 429:

                retry_after = r.headers.get(
                    "Retry-After"
                )

                if retry_after:

                    try:
                        wait = float(retry_after)
                    except ValueError:
                        wait = 2 ** attempt

                else:
                    wait = 2 ** attempt

                wait += random.uniform(1, 3)

                print(
                    f"429 rate limit "
                    f"(attempt {attempt + 1}/{max_retries})"
                )

                print(
                    f"Waiting {wait:.1f}s..."
                )

                time.sleep(wait)

                continue


            print(
                f"HTTP {r.status_code}: {url}"
            )

            return None


        except requests.RequestException as e:

            wait = (
                2 ** attempt
                + random.uniform(1, 3)
            )

            print(
                f"Request error: {e}"
            )

            print(
                f"Retrying in {wait:.1f}s..."
            )

            time.sleep(wait)


    print(
        f"Giving up: {url}"
    )

    return None


# ============================================================
# Get races
# ============================================================

race_rows = []


for year in years:

    url = (
        f"https://api.jolpi.ca/ergast/f1/"
        f"{year}/races.json"
    )

    print(
        f"Getting race calendar {year}..."
    )

    data = get_json(url)

    if data is None:
        continue


    races = (
        data
        .get("MRData", {})
        .get("RaceTable", {})
        .get("Races", [])
    )


    for race in races:

        race_rows.append({

            "season":
                int(race["season"]),

            "round":
                int(race["round"]),

            "race_name":
                race["raceName"],

            "date":
                race.get("date")
        })


    time.sleep(
        random.uniform(2, 4)
    )


races_df = pd.DataFrame(
    race_rows
)


print()
print("Races found:")
print(
    races_df[
        ["season", "round", "race_name", "date"]
    ].tail(15)
)


# ============================================================
# Download qualifying
# ============================================================

qualifying_data = []


for _, race in races_df.iterrows():

    year = int(
        race["season"]
    )

    round_num = int(
        race["round"]
    )

    race_name = race["race_name"]


    cache_file = os.path.join(
        CACHE_DIR,
        f"qualifying_{year}_{round_num}.csv"
    )


    # --------------------------------------------------------
    # Load cache
    # --------------------------------------------------------

    if os.path.exists(cache_file):

        cached = pd.read_csv(
            cache_file
        )

        qualifying_data.extend(
            cached.to_dict("records")
        )

        print(
            f"{year} R{round_num} "
            f"{race_name}: cached"
        )

        continue


    # --------------------------------------------------------
    # Request qualifying for THIS round
    # --------------------------------------------------------

    url = (
        f"https://api.jolpi.ca/ergast/f1/"
        f"{year}/{round_num}/qualifying.json"
    )


    print(
        f"{year} R{round_num} "
        f"{race_name}: downloading..."
    )


    data = get_json(url)


    if data is None:

        print(
            f"{year} R{round_num}: FAILED"
        )

        continue


    races = (
        data
        .get("MRData", {})
        .get("RaceTable", {})
        .get("Races", [])
    )


    if not races:

        print(
            f"{year} R{round_num}: "
            f"no qualifying data"
        )

        continue


    rows = []


    for race_result in races:

        for result in race_result.get(
            "QualifyingResults",
            []
        ):

            driver = result.get(
                "Driver",
                {}
            )


            rows.append({

                "season":
                    year,

                "round":
                    round_num,

                "race_name":
                    race_name,

                "position":
                    int(result["position"]),

                "driver_id":
                    driver.get(
                        "driverId"
                    ),

                "qualifying_time":
                    (
                        result.get("Q3")
                        or result.get("Q2")
                        or result.get("Q1")
                    )
            })


    if rows:

        df_round = pd.DataFrame(
            rows
        )


        # Save immediately
        df_round.to_csv(
            cache_file,
            index=False
        )


        qualifying_data.extend(
            rows
        )


        print(
            f"    {len(rows)} drivers"
        )

    else:

        print(
            f"    No qualifying results"
        )


    # Don't hammer API
    time.sleep(
        random.uniform(3, 6)
    )


# ============================================================
# Final DataFrame
# ============================================================

qualifying = pd.DataFrame(
    qualifying_data
)


if not qualifying.empty:

    qualifying.sort_values(
        ["season", "round", "position"],
        inplace=True
    )

    qualifying.reset_index(
        drop=True,
        inplace=True
    )


qualifying.to_csv(
    "qualifying_results.csv",
    index=False
)


# ============================================================
# Diagnostics
# ============================================================

print()
print("=" * 60)
print("QUALIFYING COMPLETE")
print("=" * 60)

print(
    "Shape:",
    qualifying.shape
)

print()

print(
    qualifying.groupby(
        ["season", "round"]
    ).size()
)

print()

print(
    "2026 latest qualifying:"
)

print(
    qualifying[
        qualifying["season"] == 2026
    ]
    .tail(20)
)

Getting race calendar 2021...
Getting race calendar 2022...
Getting race calendar 2023...
Getting race calendar 2024...
Getting race calendar 2025...
Getting race calendar 2026...

Races found:
     season  round                       race_name        date
122    2026      9              British Grand Prix  2026-07-05
123    2026     10              Belgian Grand Prix  2026-07-19
124    2026     11            Hungarian Grand Prix  2026-07-26
125    2026     12                Dutch Grand Prix  2026-08-23
126    2026     13              Italian Grand Prix  2026-09-06
127    2026     14              Spanish Grand Prix  2026-09-13
128    2026     15           Azerbaijan Grand Prix  2026-09-26
129    2026     16  Bahrain Grand Prix in Malaysia  2026-10-04
130    2026     17            Singapore Grand Prix  2026-10-11
131    2026     18        United States Grand Prix  2026-10-25
132    2026     19          Mexico City Grand Prix  2026-11-01
133    2026     20            Brazilian Grand Prix

In [40]:
print(type(races_df))

<class 'pandas.core.frame.DataFrame'>


In [41]:
print(len(races_df))
print(races_df.head())

137
   season  round                  race_name        date
0    2021      1         Bahrain Grand Prix  2021-03-28
1    2021      2  Emilia Romagna Grand Prix  2021-04-18
2    2021      3      Portuguese Grand Prix  2021-05-02
3    2021      4         Spanish Grand Prix  2021-05-09
4    2021      5          Monaco Grand Prix  2021-05-23


In [43]:
#weathersupposedlyfaster
import pandas as pd
import requests
import time
from datetime import datetime
from bs4 import BeautifulSoup

# ===============================
# Step 1: Fetch race URLs
# ===============================
#current_year = datetime.now().year - 1
#years = range(current_year - 5, current_year + 1)
years = range(2020, 2027)

races = []

for year in years:
    r = requests.get(f"https://api.jolpi.ca/ergast/f1/{year}/races.json")
    races_data = r.json()

    for race in races_data.get('MRData', {}).get('RaceTable', {}).get('Races', []):
        races.append({
            'season': int(race['season']),
            'round': int(race['round']),
            'circuit_id': race['Circuit']['circuitId'],
            'url': race['url']
        })

    time.sleep(0.2)

races_df = pd.DataFrame(races)

# ===============================
# Step 2: Scrape Weather
# ===============================
headers = {
    "User-Agent": "Mozilla/5.0"
}

weather_list = []

for link in races_df.url:

    weather_value = "not found"

    try:
        r = requests.get(link, headers=headers, timeout=10)

        if r.status_code != 200:
            raise ValueError(f"Bad status {r.status_code}")

        #soup = BeautifulSoup(r.text, "lxml")
        soup = BeautifulSoup(r.text, "html.parser")
        infobox = soup.select_one(".infobox")

        if infobox:
            rows = infobox.find_all("tr")

            for row in rows:
                th = row.find("th")
                td = row.find("td")

                if th and td and "weather" in th.text.lower():
                    weather_value = td.text.strip()
                    break

    except Exception as e:
        print("Error:", link, e)

    # ✅ ALWAYS append (outside try)
    weather_list.append(weather_value)

    time.sleep(0.5)

# ===============================
# Step 3: Build dataframe
# ===============================
weather = races_df[['season','round','circuit_id']].copy()
weather['weather'] = weather_list

# ===============================
# Step 4: Weather categories
# ===============================
weather_dict = {
    'weather_warm': ['clear','warm','hot','sunny','fine','mild'],
    'weather_cold': ['cold','fresh','chilly','cool'],
    'weather_dry': ['dry'],
    'weather_wet': ['showers','wet','rain','damp','thunderstorms','rainy'],
    'weather_cloudy': ['overcast','clouds','cloudy','grey']
}

weather_df = pd.DataFrame(columns=weather_dict.keys())

for col in weather_dict:
    weather_df[col] = weather['weather'].apply(
        lambda x: 1 if isinstance(x,str) and any(w in x.lower() for w in weather_dict[col]) else 0
    )

weather_info = pd.concat([weather, weather_df], axis=1)

print(weather_info.tail(20))

Error: https://en.wikipedia.org/wiki/2026_Barcelona-Catalunya Bad status 404
Error: https://en.wikipedia.org/wiki/2026_Brazilian_Grand_Prix Bad status 404
     season  round     circuit_id        weather  weather_warm  weather_cold  \
134    2026      4          miami         Cloudy             0             0   
135    2026      5     villeneuve         Cloudy             0             0   
136    2026      6         monaco          Sunny             1             0   
137    2026      7      catalunya      not found             0             0   
138    2026      8  red_bull_ring          Sunny             1             0   
139    2026      9    silverstone          Sunny             1             0   
140    2026     10            spa  Partly cloudy             0             0   
141    2026     11    hungaroring  Partly cloudy             0             0   
142    2026     12      zandvoort  Partly cloudy             0             0   
143    2026     13          monza      not fo

In [44]:
print(weather_info.tail(20))

     season  round     circuit_id        weather  weather_warm  weather_cold  \
134    2026      4          miami         Cloudy             0             0   
135    2026      5     villeneuve         Cloudy             0             0   
136    2026      6         monaco          Sunny             1             0   
137    2026      7      catalunya      not found             0             0   
138    2026      8  red_bull_ring          Sunny             1             0   
139    2026      9    silverstone          Sunny             1             0   
140    2026     10            spa  Partly cloudy             0             0   
141    2026     11    hungaroring  Partly cloudy             0             0   
142    2026     12      zandvoort  Partly cloudy             0             0   
143    2026     13          monza      not found             0             0   
144    2026     14        madring      not found             0             0   
145    2026     15           baku      n

In [45]:
weather_info.to_csv("weather_info.csv", index=False, encoding="utf-8-sig")

In [46]:
races_df.to_csv("races_df.csv", index=False, encoding="utf-8-sig")

In [47]:
print(weather_info.tail(20))

     season  round     circuit_id        weather  weather_warm  weather_cold  \
134    2026      4          miami         Cloudy             0             0   
135    2026      5     villeneuve         Cloudy             0             0   
136    2026      6         monaco          Sunny             1             0   
137    2026      7      catalunya      not found             0             0   
138    2026      8  red_bull_ring          Sunny             1             0   
139    2026      9    silverstone          Sunny             1             0   
140    2026     10            spa  Partly cloudy             0             0   
141    2026     11    hungaroring  Partly cloudy             0             0   
142    2026     12      zandvoort  Partly cloudy             0             0   
143    2026     13          monza      not found             0             0   
144    2026     14        madring      not found             0             0   
145    2026     15           baku      n

In [48]:
print(races_df.shape)
print(races_df.columns)

(154, 4)
Index(['season', 'round', 'circuit_id', 'url'], dtype='object')


In [50]:
# Convert from dict to DataFrame if needed
if isinstance(driver_standings_df, dict):
    driver_standings_df = pd.DataFrame(driver_standings)

if isinstance(constructor_standings_df, dict):
    constructor_standings_df = pd.DataFrame(constructor_standings)

In [51]:
results_df.to_csv("results_df.csv", index=False, encoding="utf-8-sig")

In [52]:
qualifying.columns = qualifying.columns.str.strip().str.lower()

In [53]:
print(list(qualifying.columns))

['season', 'round', 'race_name', 'position', 'driver_id', 'qualifying_time']


In [54]:
qualifying.columns = [
    "season",
    "round",
    "race_name",
    "position",
    "driver_id",
    "qualifying_time"
]

In [55]:
print(qualifying.tail(40))
print(qualifying.columns)

      season  round           race_name  position       driver_id  \
2520    2026     12    Dutch Grand Prix         5        hamilton   
2521    2026     12    Dutch Grand Prix         6         leclerc   
2522    2026     12    Dutch Grand Prix         7  max_verstappen   
2523    2026     12    Dutch Grand Prix         8          lawson   
2524    2026     12    Dutch Grand Prix         9       bortoleto   
2525    2026     12    Dutch Grand Prix        10  arvid_lindblad   
2526    2026     12    Dutch Grand Prix        11           gasly   
2527    2026     12    Dutch Grand Prix        12         tsunoda   
2528    2026     12    Dutch Grand Prix        13      hulkenberg   
2529    2026     12    Dutch Grand Prix        14       colapinto   
2530    2026     12    Dutch Grand Prix        15            ocon   
2531    2026     12    Dutch Grand Prix        16           albon   
2532    2026     12    Dutch Grand Prix        17           sainz   
2533    2026     12    Dutch Grand

In [56]:
#merging
import pandas as pd

# --- Step 0: Standardize column names ---
results_df = results_df.rename(columns={"circuit": "circuit_id"})
driver_standings_df = driver_standings_df.rename(columns={"driver_id": "driver"})
constructor_standings_df = constructor_standings_df.rename(columns={"constructor_id": "constructor"})

# --- Step 1: Normalize circuit_id ---
for df in [races_df, races_df, weather_info, results_df]:
    if 'circuit_id' in df.columns:
        df['circuit_id'] = df['circuit_id'].str.strip().str.lower()

# --- Step 2: Normalize driver ---
for df in [driver_standings_df, results_df, qualifying]:
    if 'driver' in df.columns:
        df['driver'] = df['driver'].str.strip().str.lower()

# --- Step 3: Normalize constructor ---
for df in [constructor_standings_df, results_df]:
    if 'constructor' in df.columns:
        df['constructor'] = df['constructor'].str.strip().str.lower()

# --- Step 4: Merge pipeline with controlled suffixes ---

# Races + Weather
df1 = pd.merge(races_df, weather_info, how='left',
               on=['season', 'round', 'circuit_id'])
print("After merging races + weather:", df1.shape)

df1.to_csv("df1.csv", index=False, encoding="utf-8-sig")


# Results
df2 = pd.merge(df1, results_df, how='left',
              # on=['season', 'round', 'circuit_id'],
                 on=['season', 'round'],  
               suffixes=('', '_res'))
print("After merging results:", df2.shape)

df2.to_csv("df2.csv", index=False, encoding="utf-8-sig")

# Driver standings
df3 = pd.merge(df2, driver_standings_df, how='left',
               on=['season', 'round', 'driver'],
               suffixes=('', '_drv'))
print("After merging driver standings:", df3.shape)

df3.to_csv("df3.csv", index=False, encoding="utf-8-sig")

# Constructor standings
df4 = pd.merge(df3, constructor_standings_df, how='left',
               on=['season', 'round', 'constructor'],
               suffixes=('', '_cons'))
print("After merging constructor standings:", df4.shape)

df4.to_csv("df4.csv", index=False, encoding="utf-8-sig")

# Ensure 'position' is of the same type in both dataframes
df4['position'] = df4['position'].astype(str)  # or .astype(int) based on your preference
qualifying['position'] = qualifying['position'].astype(str)  # or .astype(int)



After merging races + weather: (154, 10)
After merging results: (2570, 21)
After merging driver standings: (2570, 24)
After merging constructor standings: (2570, 30)


In [57]:
# Check unique values in both dataframes for position, season, and round
print(df4[['season', 'round', 'position', 'driver']].drop_duplicates().tail(20))
print(qualifying[['season', 'round', 'position', 'driver_id']].drop_duplicates().tail(40))

      season  round position          driver
2550    2026     12     14.0       colapinto
2551    2026     12     15.0           perez
2552    2026     12     16.0           sainz
2553    2026     12     17.0           albon
2554    2026     12     18.0          bottas
2555    2026     12     19.0            ocon
2556    2026     12     20.0          stroll
2557    2026     12     21.0         bearman
2558    2026     12     22.0  max_verstappen
2559    2026     13      nan             NaN
2560    2026     14      nan             NaN
2561    2026     15      nan             NaN
2562    2026     16      nan             NaN
2563    2026     17      nan             NaN
2564    2026     18      nan             NaN
2565    2026     19      nan             NaN
2566    2026     20      nan             NaN
2567    2026     21      nan             NaN
2568    2026     22      nan             NaN
2569    2026     23      nan             NaN
      season  round position       driver_id
2520    20

In [58]:
# Check rows that do not have matching 'position', 'season', and 'round'
df4_not_in_qualifying = df4[~df4[['season', 'round', 'position']].apply(tuple, axis=1).isin(qualifying[['season', 'round', 'position']].apply(tuple, axis=1))]
print(df4_not_in_qualifying[['season', 'round', 'position']].tail(50))


      season  round position
2520    2026     11      6.0
2521    2026     11      7.0
2522    2026     11      8.0
2523    2026     11      9.0
2524    2026     11     10.0
2525    2026     11     11.0
2526    2026     11     12.0
2527    2026     11     13.0
2528    2026     11     14.0
2529    2026     11     15.0
2530    2026     11     16.0
2531    2026     11     17.0
2532    2026     11     18.0
2533    2026     11     19.0
2534    2026     11     20.0
2535    2026     11     21.0
2536    2026     11     22.0
2537    2026     12      1.0
2538    2026     12      2.0
2539    2026     12      3.0
2540    2026     12      4.0
2541    2026     12      5.0
2542    2026     12      6.0
2543    2026     12      7.0
2544    2026     12      8.0
2545    2026     12      9.0
2546    2026     12     10.0
2547    2026     12     11.0
2548    2026     12     12.0
2549    2026     12     13.0
2550    2026     12     14.0
2551    2026     12     15.0
2552    2026     12     16.0
2553    2026  

In [59]:
# Qualifying
final_df = pd.merge(df4, qualifying, how='left',
                    on=['season', 'round', 'position'],
                    suffixes=('', '_qual'))
print("After merging qualifying:", final_df.shape)

# --- Step 5: Clean up column names ---
# Remove unwanted duplicates or columns ending with _x/_y/_dup
final_df = final_df.loc[:, ~final_df.columns.str.endswith(('_dup', '_res', '_drv', '_cons', '_qual'))]

final_df['qualifying_time'] = final_df['qualifying_time'].fillna(0)
final_df = final_df.fillna(0)

# Optional sanity check
print(final_df[['race_name', 'position', 'driver', 'qualifying_time']].head(40))

final_df.to_csv("final_df.csv", index=False, encoding="utf-8-sig")

After merging qualifying: (2570, 33)
                    race_name position           driver  qualifying_time
0                           0      nan                0                0
1                           0      nan                0                0
2                           0      nan                0                0
3                           0      nan                0                0
4                           0      nan                0                0
5                           0      nan                0                0
6                           0      nan                0                0
7                           0      nan                0                0
8                           0      nan                0                0
9                           0      nan                0                0
10                          0      nan                0                0
11                          0      nan                0                0
12            

C:\Users\RObus\AppData\Local\Temp\ipykernel_1996\2296409025.py:11: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  final_df['qualifying_time'] = final_df['qualifying_time'].fillna(0)
C:\Users\RObus\AppData\Local\Temp\ipykernel_1996\2296409025.py:12: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  final_df = final_df.fillna(0)


In [60]:
# Check unmatched qualifying rows
merged_check = pd.merge(df4, qualifying, how='left',
                        #on=['season', 'round', 'driver'],
                        on=['season', 'round'],
                        indicator=True)

unmatched = merged_check[merged_check['_merge'] == 'left_only']
print("Number of unmatched rows in df4:", unmatched.shape[0])
print(unmatched[['season', 'round', 'driver']].tail(40))

Number of unmatched rows in df4: 27
       season  round driver
0        2020      1    NaN
1        2020      2    NaN
2        2020      3    NaN
3        2020      4    NaN
4        2020      5    NaN
5        2020      6    NaN
6        2020      7    NaN
7        2020      8    NaN
8        2020      9    NaN
9        2020     10    NaN
10       2020     11    NaN
11       2020     12    NaN
12       2020     13    NaN
13       2020     14    NaN
14       2020     15    NaN
15       2020     16    NaN
16       2020     17    NaN
51282    2026     14    NaN
51283    2026     15    NaN
51284    2026     16    NaN
51285    2026     17    NaN
51286    2026     18    NaN
51287    2026     19    NaN
51288    2026     20    NaN
51289    2026     21    NaN
51290    2026     22    NaN
51291    2026     23    NaN


In [61]:
print("df1 columns:", df1.columns.tolist())
print("results_df columns:", results_df.columns.tolist())


df1 columns: ['season', 'round', 'circuit_id', 'url', 'weather', 'weather_warm', 'weather_cold', 'weather_dry', 'weather_wet', 'weather_cloudy']
results_df columns: ['season', 'round', 'date', 'race_name', 'circuit_id', 'driver', 'driver_name', 'constructor', 'constructor_name', 'grid', 'position', 'status', 'points']


In [62]:
print(final_df.columns.tolist())

['season', 'round', 'circuit_id', 'url', 'weather', 'weather_warm', 'weather_cold', 'weather_dry', 'weather_wet', 'weather_cloudy', 'date', 'race_name', 'driver', 'driver_name', 'constructor', 'constructor_name', 'grid', 'position', 'status', 'points', 'driver_points', 'driver_wins', 'driver_standings_pos', 'constructor_points', 'constructor_wins', 'constructor_standings_pos', 'constructor_points_before', 'constructor_wins_before', 'constructor_standings_pos_before', 'driver_id', 'qualifying_time']


In [63]:
print("races_df columns:", races_df.columns.tolist())
print("weather_info columns:", weather_info.columns.tolist())
print("results_df columns:", results_df.columns.tolist())
print("driver_standings columns:", driver_standings_df.columns.tolist())
print("constructor_standings:", constructor_standings_df.columns.tolist())
print("qualifying:", qualifying.columns.tolist())

races_df columns: ['season', 'round', 'circuit_id', 'url']
weather_info columns: ['season', 'round', 'circuit_id', 'weather', 'weather_warm', 'weather_cold', 'weather_dry', 'weather_wet', 'weather_cloudy']
results_df columns: ['season', 'round', 'date', 'race_name', 'circuit_id', 'driver', 'driver_name', 'constructor', 'constructor_name', 'grid', 'position', 'status', 'points']
driver_standings columns: ['season', 'round', 'driver', 'driver_points', 'driver_wins', 'driver_standings_pos']
constructor_standings: ['season', 'round', 'constructor', 'constructor_points', 'constructor_wins', 'constructor_standings_pos', 'constructor_points_before', 'constructor_wins_before', 'constructor_standings_pos_before']
qualifying: ['season', 'round', 'race_name', 'position', 'driver_id', 'qualifying_time']


In [64]:
print(type(constructor_standings_df))

<class 'pandas.core.frame.DataFrame'>


In [65]:
print (final_df.tail(20))

      season  round  circuit_id  \
2550    2026     12   zandvoort   
2551    2026     12   zandvoort   
2552    2026     12   zandvoort   
2553    2026     12   zandvoort   
2554    2026     12   zandvoort   
2555    2026     12   zandvoort   
2556    2026     12   zandvoort   
2557    2026     12   zandvoort   
2558    2026     12   zandvoort   
2559    2026     13       monza   
2560    2026     14     madring   
2561    2026     15        baku   
2562    2026     16      sepang   
2563    2026     17  marina_bay   
2564    2026     18    americas   
2565    2026     19   rodriguez   
2566    2026     20  interlagos   
2567    2026     21       vegas   
2568    2026     22      losail   
2569    2026     23  yas_marina   

                                                    url        weather  \
2550  https://en.wikipedia.org/wiki/2026_Dutch_Grand...  Partly cloudy   
2551  https://en.wikipedia.org/wiki/2026_Dutch_Grand...  Partly cloudy   
2552  https://en.wikipedia.org/wiki/2026_D

In [66]:
print(final_df.columns)

Index(['season', 'round', 'circuit_id', 'url', 'weather', 'weather_warm',
       'weather_cold', 'weather_dry', 'weather_wet', 'weather_cloudy', 'date',
       'race_name', 'driver', 'driver_name', 'constructor', 'constructor_name',
       'grid', 'position', 'status', 'points', 'driver_points', 'driver_wins',
       'driver_standings_pos', 'constructor_points', 'constructor_wins',
       'constructor_standings_pos', 'constructor_points_before',
       'constructor_wins_before', 'constructor_standings_pos_before',
       'driver_id', 'qualifying_time'],
      dtype='object')


In [67]:
# Convert position to numeric, coerce errors (invalid entries become NaN)
final_df['position'] = pd.to_numeric(final_df['position'], errors='coerce')

# Optional: replace NaN with a large number (so they are not considered top3)
final_df['position'] = final_df['position'].fillna(999)

# Now create the target
#final_df['top3'] = (final_df['position'] <= 3).astype(int)

In [68]:
# Check the unique values of the columns
print(final_df['driver'].unique())
print(final_df['constructor'].unique())
print(final_df['race_name'].unique())

[0 'hamilton' 'max_verstappen' 'bottas' 'norris' 'perez' 'leclerc'
 'ricciardo' 'sainz' 'tsunoda' 'stroll' 'raikkonen' 'giovinazzi' 'ocon'
 'russell' 'vettel' 'mick_schumacher' 'gasly' 'latifi' 'alonso' 'mazepin'
 'kubica' 'kevin_magnussen' 'zhou' 'albon' 'hulkenberg' 'de_vries'
 'sargeant' 'piastri' 'lawson' 'bearman' 'colapinto' 'doohan' 'antonelli'
 'bortoleto' 'hadjar' 'arvid_lindblad']
[0 'mercedes' 'red_bull' 'mclaren' 'ferrari' 'alphatauri' 'aston_martin'
 'alfa' 'alpine' 'williams' 'haas' 'sauber' 'rb' 'audi' 'cadillac']
[0 'Bahrain Grand Prix' 'Emilia Romagna Grand Prix'
 'Portuguese Grand Prix' 'Spanish Grand Prix' 'Monaco Grand Prix'
 'Azerbaijan Grand Prix' 'French Grand Prix' 'Styrian Grand Prix'
 'Austrian Grand Prix' 'British Grand Prix' 'Hungarian Grand Prix'
 'Belgian Grand Prix' 'Dutch Grand Prix' 'Italian Grand Prix'
 'Russian Grand Prix' 'Turkish Grand Prix' 'United States Grand Prix'
 'Mexico City Grand Prix' 'São Paulo Grand Prix' 'Qatar Grand Prix'
 'Saudi Arabia

In [69]:
print(final_df.dtypes)

season                                int64
round                                 int64
circuit_id                           object
url                                  object
weather                              object
weather_warm                          int64
weather_cold                          int64
weather_dry                           int64
weather_wet                           int64
weather_cloudy                        int64
date                                 object
race_name                            object
driver                               object
driver_name                          object
constructor                          object
constructor_name                     object
grid                                float64
position                            float64
status                               object
points                              float64
driver_points                       float64
driver_wins                         float64
driver_standings_pos            

In [70]:
from sklearn.preprocessing import LabelEncoder

In [71]:
# Convert columns to strings and handle NaNs
final_df['driver'] = final_df['driver'].fillna('missing').astype(str)
final_df['constructor'] = final_df['constructor'].fillna('missing').astype(str)
final_df['circuit_id'] = final_df['circuit_id'].fillna('missing').astype(str)

# Verify that all values are now strings
print(final_df[['driver', 'constructor', 'circuit_id']].head())

# Now apply LabelEncoder to the cleaned columns
le_driver = LabelEncoder()
final_df['driver_enc'] = le_driver.fit_transform(final_df['driver'])

le_constructor = LabelEncoder()
final_df['constructor_enc'] = le_constructor.fit_transform(final_df['constructor'])

le_circuit = LabelEncoder()
final_df['circuit_enc'] = le_circuit.fit_transform(final_df['circuit_id'])

  driver constructor     circuit_id
0      0           0  red_bull_ring
1      0           0  red_bull_ring
2      0           0    hungaroring
3      0           0    silverstone
4      0           0    silverstone


In [72]:
print(final_df.dtypes)

season                                int64
round                                 int64
circuit_id                           object
url                                  object
weather                              object
weather_warm                          int64
weather_cold                          int64
weather_dry                           int64
weather_wet                           int64
weather_cloudy                        int64
date                                 object
race_name                            object
driver                               object
driver_name                          object
constructor                          object
constructor_name                     object
grid                                float64
position                            float64
status                               object
points                              float64
driver_points                       float64
driver_wins                         float64
driver_standings_pos            

In [73]:
final_df['top3'] = final_df['position'].apply(lambda x: 1 if x <= 3 else 0)

In [74]:
#HistGradientBoostingClassifier
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import HistGradientBoostingClassifier

# Step 1: Fill missing numeric values for columns that exist
numeric_features = [
    'grid', 'driver_points', 'driver_wins', 'driver_standings_pos',
    'constructor_points', 'constructor_wins', 'constructor_standings_pos'
]

final_df[numeric_features] = final_df[numeric_features].fillna(0)

# Step 2: Encode categorical variables
from sklearn.preprocessing import LabelEncoder

le_driver = LabelEncoder()
final_df['driver_enc'] = le_driver.fit_transform(final_df['driver'])

le_constructor = LabelEncoder()
final_df['constructor_enc'] = le_constructor.fit_transform(final_df['constructor'])

le_circuit = LabelEncoder()
final_df['circuit_enc'] = le_circuit.fit_transform(final_df['circuit_id'])

# Step 3: Compute rolling / historical features (e.g., last 5 races, circuit avg)
# Example for last 5 races average finish
final_df['driver_avg_finish_last5'] = final_df.groupby('driver')['position'].transform(lambda x: x.rolling(5, min_periods=1).mean())
# Example for circuit average
final_df['driver_circuit_avg_finish'] = final_df.groupby(['driver','circuit_id'])['position'].transform('mean')

# Now all features exist
features = [
    'grid', 'driver_points', 'driver_wins', 'driver_standings_pos',
    'driver_avg_finish_last5', 'driver_circuit_avg_finish',
    'constructor_points', 'constructor_wins', 'constructor_standings_pos',
    'driver_enc', 'constructor_enc', 'circuit_enc'
]

# Fill any remaining NaNs (from rolling averages)
final_df[features] = final_df[features].fillna(0)


# -------------------------
# 1️⃣ Prepare features & target
# -------------------------
features = [
    'grid', 'driver_points', 'driver_wins', 'driver_standings_pos',
    'driver_avg_finish_last5', 'driver_circuit_avg_finish',
    'constructor_points', 'constructor_wins', 'constructor_standings_pos',
    'driver_enc', 'constructor_enc', 'circuit_enc'
]

# Fill missing numeric values in historical data
final_df[features] = final_df[features].fillna(0)


# -------------------------
# 2️⃣ Encode drivers, constructors, circuits
# Ensure all values in 'driver' are treated as strings, even if they are numbers
final_df['driver'] = final_df['driver'].apply(str)

# Repeat for other columns if necessary
final_df['constructor'] = final_df['constructor'].apply(str)
final_df['circuit'] = final_df['circuit_id'].apply(str)

# Now apply the LabelEncoder
le_driver = LabelEncoder()
final_df['driver_enc'] = le_driver.fit_transform(final_df['driver'])

le_constructor = LabelEncoder()
final_df['constructor_enc'] = le_constructor.fit_transform(final_df['constructor'])

le_circuit = LabelEncoder()
final_df['circuit_enc'] = le_circuit.fit_transform(final_df['circuit_id'])

# -------------------------
# 3️⃣ Split training / validation
# -------------------------
train = final_df[final_df['season'] < 2025]        # 2021–2024
validate = final_df[final_df['season'] == 2025]   # 2025
#validate = final_df[final_df['season'].isin([2024, 2025])]

X_train = train[features]
y_train = train['top3']

X_validate = validate[features]
y_validate = validate['top3']

# -------------------------
# 4️⃣ Train model
# -------------------------
model = HistGradientBoostingClassifier(max_iter=500, learning_rate=0.03, max_depth=8)
model.fit(X_train, y_train)

# Validate
y_pred_val = model.predict(X_validate)
from sklearn.metrics import classification_report
print("2025 Validation Report:")
print(classification_report(y_validate, y_pred_val))

# -------------------------
# 5️⃣ Prepare 2026 features from 2025 stats
# -------------------------
# Latest driver stats
driver_stats_2025 = final_df[final_df['season'] == 2025].groupby('driver').agg({
    'driver_points':'sum','driver_wins':'sum','driver_standings_pos':'min',
    'driver_avg_finish_last5':'mean','driver_circuit_avg_finish':'mean'
}).reset_index()

# Latest constructor stats
constructor_stats_2025 = final_df[final_df['season'] == 2025].groupby('constructor').agg({
    'constructor_points':'sum','constructor_wins':'sum','constructor_standings_pos':'min'
}).reset_index()

# Circuits in 2026
circuits_2026 = final_df[final_df['season']==2026]['circuit_id'].unique()
drivers_2025 = driver_stats_2025['driver'].unique()

# Build 2026 driver x circuit combinations
rows_2026 = []
for round_, circuit in enumerate(circuits_2026, start=1):
    for driver in drivers_2025:
        constructor = final_df[(final_df['season']==2025) & (final_df['driver']==driver)]['constructor'].iloc[0]
        driver_row = driver_stats_2025[driver_stats_2025['driver']==driver].iloc[0]
        constructor_row = constructor_stats_2025[constructor_stats_2025['constructor']==constructor].iloc[0]

        rows_2026.append({
            'season':2026,'round':round_,'circuit_id':circuit,
            'driver':driver,'constructor':constructor,
            'grid':0,
            'driver_points':driver_row['driver_points'],
            'driver_wins':driver_row['driver_wins'],
            'driver_standings_pos':driver_row['driver_standings_pos'],
            'driver_avg_finish_last5':driver_row['driver_avg_finish_last5'],
            'driver_circuit_avg_finish':driver_row['driver_circuit_avg_finish'],
            'constructor_points':constructor_row['constructor_points'],
            'constructor_wins':constructor_row['constructor_wins'],
            'constructor_standings_pos':constructor_row['constructor_standings_pos'],
        })

df_2026_features = pd.DataFrame(rows_2026)

# Encode drivers / constructors / circuits for 2026
df_2026_features['driver_enc'] = le_driver.transform(df_2026_features['driver'])
df_2026_features['constructor_enc'] = le_constructor.transform(df_2026_features['constructor'])
df_2026_features['circuit_enc'] = le_circuit.transform(df_2026_features['circuit_id'])

# -------------------------
# 6️⃣ Predict Top-3 probabilities for 2026
# -------------------------
X_2026 = df_2026_features[features]
df_2026_features['top3_prob'] = model.predict_proba(X_2026)[:,1]

# Select top 3 drivers per GP
predicted_podium_2026_hist = (
    df_2026_features
    .sort_values(['round','top3_prob'], ascending=[True, False])
    .groupby('round')
    .head(3)
)

print(predicted_podium_2026_hist[['round','driver','constructor','top3_prob']])

predicted_podium_2026_hist.to_csv("predicted_podium_hist_2026.csv", index=False, encoding="utf-8-sig")


2025 Validation Report:
              precision    recall  f1-score   support

           0       0.96      0.96      0.96       407
           1       0.78      0.78      0.78        72

    accuracy                           0.93       479
   macro avg       0.87      0.87      0.87       479
weighted avg       0.93      0.93      0.93       479

     round          driver constructor  top3_prob
12       1         leclerc     ferrari   0.120846
13       1  max_verstappen    red_bull   0.091116
2        1       antonelli    mercedes   0.040819
33       2         leclerc     ferrari   0.224242
34       2  max_verstappen    red_bull   0.151018
..     ...             ...         ...        ...
454     22  max_verstappen    red_bull   0.111937
443     22       antonelli    mercedes   0.038028
474     23         leclerc     ferrari   0.347904
475     23  max_verstappen    red_bull   0.246867
478     23         piastri     mclaren   0.150884

[69 rows x 4 columns]


In [75]:
print(df_2026_features.columns)
print(df_2026_features.head())
print(len(df_2026_features))

Index(['season', 'round', 'circuit_id', 'driver', 'constructor', 'grid',
       'driver_points', 'driver_wins', 'driver_standings_pos',
       'driver_avg_finish_last5', 'driver_circuit_avg_finish',
       'constructor_points', 'constructor_wins', 'constructor_standings_pos',
       'driver_enc', 'constructor_enc', 'circuit_enc', 'top3_prob'],
      dtype='object')
   season  round   circuit_id     driver   constructor  grid  driver_points  \
0    2026      1  albert_park      albon      williams     0         1186.0   
1    2026      1  albert_park     alonso  aston_martin     0          466.0   
2    2026      1  albert_park  antonelli      mercedes     0         1681.0   
3    2026      1  albert_park    bearman          haas     0          360.0   
4    2026      1  albert_park  bortoleto        sauber     0          209.0   

   driver_wins  driver_standings_pos  driver_avg_finish_last5  \
0          0.0                   5.0                11.083333   
1          0.0             

In [76]:
#print(df_2026_features.columns)

In [77]:
#histgradientweatherandqualyfing
import pandas as pd 
from sklearn.preprocessing import LabelEncoder 
from sklearn.ensemble import HistGradientBoostingClassifier 
# -------------------------
# Qualifying Features
# -------------------------

# Convert qualifying time to numeric seconds if needed
final_df['qualifying_time'] = pd.to_numeric(
    final_df['qualifying_time'],
    errors='coerce'
)

# Pole time for each race
final_df['pole_time'] = (
    final_df.groupby(['season', 'round'])['qualifying_time']
    .transform('min')
)

# Gap to pole
final_df['qualifying_gap'] = (
    final_df['qualifying_time']
    - final_df['pole_time']
)

# Driver qualifying form (previous 5 races)
final_df['avg_qualifying_gap_last5'] = (
    final_df.groupby('driver')['qualifying_gap']
    .transform(
        lambda x: x.shift(1)
                  .rolling(5, min_periods=1)
                  .mean()
    )
)


# Step 1: Fill missing numeric values for columns that exist 
#numeric_features = [ 'grid', 'driver_points', 'driver_wins', 'driver_standings_pos', 'constructor_points', 'constructor_wins',
                     #'constructor_standings_pos', 'weather_warm', 'weather_cold', 'weather_dry', 'weather_wet', 'weather_cloudy' ]
numeric_features = [
    'grid',
    'qualifying_time',
    'qualifying_gap',
    'avg_qualifying_gap_last5',
    'driver_points',
    'driver_wins',
    'driver_standings_pos',
    'constructor_points',
    'constructor_wins',
    'constructor_standings_pos',
    'weather_warm',
    'weather_cold',
    'weather_dry',
    'weather_wet',
    'weather_cloudy'
]
# Fill missing values for numeric features, including weather 
final_df[numeric_features] = final_df[numeric_features].fillna(0) 
# Step 2: Encode categorical variables 
le_driver = LabelEncoder()
final_df['driver_enc'] = le_driver.fit_transform(final_df['driver']) 
le_constructor = LabelEncoder() 
final_df['constructor_enc'] = le_constructor.fit_transform(final_df['constructor']) 
le_circuit = LabelEncoder() 
final_df['circuit_enc'] = le_circuit.fit_transform(final_df['circuit_id']) 
# Step 3: Compute rolling / historical features (e.g., last 5 races, circuit avg) 
final_df['driver_avg_finish_last5'] = final_df.groupby('driver')['position'].transform(lambda x: x.rolling(5, min_periods=1).mean())
final_df['driver_circuit_avg_finish'] = final_df.groupby(['driver', 'circuit_id'])['position'].transform('mean') 
# Step 4: Prepare feature set with weather features included 
features = [ 'grid', 'driver_points', 'driver_wins', 'driver_standings_pos',
'driver_avg_finish_last5', 'driver_circuit_avg_finish', 'constructor_points', 'constructor_wins', 'constructor_standings_pos',
             'driver_enc', 'constructor_enc', 'circuit_enc', 'weather_warm', 'weather_cold', 'weather_dry', 'weather_wet', 'weather_cloudy' ]
# Fill any remaining NaNs (from rolling averages and weather features) 
final_df[features] = final_df[features].fillna(0)
# ------------------------- # 1️⃣ Prepare features & target # ------------------------- 
#features = [ 'grid', 'driver_points', 'driver_wins', 'driver_standings_pos', 'driver_avg_finish_last5', 'driver_circuit_avg_finish',
             #'constructor_points', 'constructor_wins', 'constructor_standings_pos', 'driver_enc', 'constructor_enc', 'circuit_enc',
             #'weather_warm', 'weather_cold', 'weather_dry', 'weather_wet', 'weather_cloudy' ]
features = [
    'grid',

    'qualifying_time',
    'qualifying_gap',
    'avg_qualifying_gap_last5',

    'driver_points',
    'driver_wins',
    'driver_standings_pos',

    'driver_avg_finish_last5',
    'driver_circuit_avg_finish',

    'constructor_points',
    'constructor_wins',
    'constructor_standings_pos',

    'driver_enc',
    'constructor_enc',
    'circuit_enc',

    'weather_warm',
    'weather_cold',
    'weather_dry',
    'weather_wet',
    'weather_cloudy'
]

# Fill missing numeric values in historical data 
final_df[features] = final_df[features].fillna(0) 
# ------------------------- # 2️⃣ Encode drivers, constructors, circuits # ------------------------- 
final_df['driver'] = final_df['driver'].apply(str) 
final_df['constructor'] = final_df['constructor'].apply(str)
final_df['circuit'] = final_df['circuit_id'].apply(str)
le_driver = LabelEncoder() 
final_df['driver_enc'] = le_driver.fit_transform(final_df['driver']) 
le_constructor = LabelEncoder()
final_df['constructor_enc'] = le_constructor.fit_transform(final_df['constructor']) 
le_circuit = LabelEncoder() 
final_df['circuit_enc'] = le_circuit.fit_transform(final_df['circuit_id']) 
# ------------------------- # 3️⃣ Split training / validation # ------------------------- 
train = final_df[final_df['season'] < 2025] 
# 2021–2024 
validate = final_df[final_df['season'] == 2025] 
# 2025 
X_train = train[features] 
y_train = train['top3']
X_validate = validate[features] 
y_validate = validate['top3'] 
# ------------------------- # 4️⃣ Train model # ------------------------- 
model = HistGradientBoostingClassifier(max_iter=500, learning_rate=0.05, max_depth=6) 
model.fit(X_train, y_train) 
# Validate
y_pred_val = model.predict(X_validate)
from sklearn.metrics import classification_report 
print("2025 Validation Report:") 
print(classification_report(y_validate, y_pred_val)) 

# 5️⃣ Prepare 2026 features from 2025 stats
# -------------------------

driver_stats_2025 = (
    final_df[final_df['season'] == 2025]
    .groupby('driver')
    .agg({
        'driver_points': 'sum',
        'driver_wins': 'sum',
        'driver_standings_pos': 'min',
        'driver_avg_finish_last5': 'mean',
        'driver_circuit_avg_finish': 'mean',
        'qualifying_time': 'mean',
        'qualifying_gap': 'mean',
        'avg_qualifying_gap_last5': 'mean'
    })
    .reset_index()
)




constructor_stats_2025 = (
    final_df[final_df['season'] == 2025]
    .groupby('constructor')
    .agg({
        'constructor_points': 'sum',
        'constructor_wins': 'sum',
        'constructor_standings_pos': 'min'
    })
    .reset_index()
)

circuits_2026 = final_df.loc[
    final_df['season'] == 2026,
    'circuit_id'
].unique()

drivers_2025 = driver_stats_2025['driver'].unique()

rows_2026 = []

for round_, circuit in enumerate(circuits_2026, start=1):

    # Get weather for this circuit from historical races
    subset = final_df[
        (final_df['season'] == 2025) &
        (final_df['circuit_id'] == circuit)
    ]

    if subset.empty:
        weather_values = {
            'weather_warm': 0,
            'weather_cold': 0,
            'weather_dry': 1,
            'weather_wet': 0,
            'weather_cloudy': 0
        }
    else:
        weather_row = subset.iloc[0]

        weather_values = {
            'weather_warm': weather_row['weather_warm'],
            'weather_cold': weather_row['weather_cold'],
            'weather_dry': weather_row['weather_dry'],
            'weather_wet': weather_row['weather_wet'],
            'weather_cloudy': weather_row['weather_cloudy']
        }

    # LOOP OVER ALL DRIVERS
    for driver in drivers_2025:

        constructor = (
            final_df[
                (final_df['season'] == 2025) &
                (final_df['driver'] == driver)
            ]['constructor']
            .iloc[0]
        )

        driver_row = (
            driver_stats_2025[
                driver_stats_2025['driver'] == driver
            ]
            .iloc[0]
        )

        constructor_row = (
            constructor_stats_2025[
                constructor_stats_2025['constructor'] == constructor
            ]
            .iloc[0]
        )

        rows_2026.append({
            'season': 2026,
            'round': round_,
            'circuit_id': circuit,

            'driver': driver,
            'constructor': constructor,

            # placeholder qualifying position
            'grid': 10,

            'driver_points': driver_row['driver_points'],
            'driver_wins': driver_row['driver_wins'],
            'driver_standings_pos': driver_row['driver_standings_pos'],
            'driver_avg_finish_last5': driver_row['driver_avg_finish_last5'],
            'driver_circuit_avg_finish': driver_row['driver_circuit_avg_finish'],

            'constructor_points': constructor_row['constructor_points'],
            'constructor_wins': constructor_row['constructor_wins'],
            'constructor_standings_pos': constructor_row['constructor_standings_pos'],
            'qualifying_time': driver_row['qualifying_time'],
            'qualifying_gap': driver_row['qualifying_gap'],
            'avg_qualifying_gap_last5': driver_row['avg_qualifying_gap_last5'],

            **weather_values
        })

df_2026_features = pd.DataFrame(rows_2026)

print("Rows:", len(df_2026_features))
print("Unique drivers:", df_2026_features['driver'].nunique())

# Encode drivers / constructors / circuits for 2026 
df_2026_features['driver_enc'] = le_driver.transform(df_2026_features['driver']) 
df_2026_features['constructor_enc'] = le_constructor.transform(df_2026_features['constructor'])
df_2026_features['circuit_enc'] = le_circuit.transform(df_2026_features['circuit_id']) 
# ------------------------- # 6️⃣ Predict Top-3 probabilities for 2026 # ------------------------- 
X_2026 = df_2026_features[features] 
df_2026_features['top3_prob'] = model.predict_proba(X_2026)[:,1]
# Select top 3 drivers per GP 
predicted_podium_2026 = ( df_2026_features .sort_values(['round', 'top3_prob'], ascending=[True, False]) .groupby('round') .head(3) )
print(predicted_podium_2026[['round','driver','constructor','top3_prob']]) 
predicted_podium_2026.to_csv("predicted_podium_2026_W.csv", index=False, encoding="utf-8-sig")



2025 Validation Report:
              precision    recall  f1-score   support

           0       0.97      0.97      0.97       407
           1       0.81      0.81      0.81        72

    accuracy                           0.94       479
   macro avg       0.89      0.89      0.89       479
weighted avg       0.94      0.94      0.94       479

Rows: 483
Unique drivers: 21
     round          driver   constructor  top3_prob
9        1        hamilton       ferrari   0.089004
1        1          alonso  aston_martin   0.071086
12       1         leclerc       ferrari   0.030318
30       2        hamilton       ferrari   0.065247
34       2  max_verstappen      red_bull   0.042638
..     ...             ...           ...        ...
453     22         leclerc       ferrari   0.017275
454     22  max_verstappen      red_bull   0.014378
471     23        hamilton       ferrari   0.181510
474     23         leclerc       ferrari   0.061231
475     23  max_verstappen      red_bull   0.035

In [78]:
print(driver_stats_2025.shape)
print(driver_stats_2025.head())
print(driver_stats_2025['driver'].nunique())

(21, 9)
      driver  driver_points  driver_wins  driver_standings_pos  \
0      albon         1186.0          0.0                   5.0   
1     alonso          466.0          0.0                   0.0   
2  antonelli         1681.0          0.0                   4.0   
3    bearman          360.0          0.0                  11.0   
4  bortoleto          209.0          0.0                   0.0   

   driver_avg_finish_last5  driver_circuit_avg_finish  qualifying_time  \
0                11.083333                  13.295833              0.0   
1                11.975000                  10.698611              0.0   
2                 9.545139                   7.937500              0.0   
3                11.556250                  12.208333              0.0   
4                14.428472                  13.791667              0.0   

   qualifying_gap  avg_qualifying_gap_last5  
0             0.0                       0.0  
1             0.0                       0.0  
2           

In [79]:
#randomforest
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# -------------------------
# Step 1: Fill missing numeric values
# -------------------------
numeric_features = [
    'grid', 'driver_points', 'driver_wins', 'driver_standings_pos',
    'constructor_points', 'constructor_wins', 'constructor_standings_pos'
]

final_df[numeric_features] = final_df[numeric_features].fillna(0)

# -------------------------
# Step 2: Encode categorical variables
# -------------------------
final_df['driver'] = final_df['driver'].astype(str)
final_df['constructor'] = final_df['constructor'].astype(str)
final_df['circuit_id'] = final_df['circuit_id'].astype(str)

le_driver = LabelEncoder()
final_df['driver_enc'] = le_driver.fit_transform(final_df['driver'])

le_constructor = LabelEncoder()
final_df['constructor_enc'] = le_constructor.fit_transform(final_df['constructor'])

le_circuit = LabelEncoder()
final_df['circuit_enc'] = le_circuit.fit_transform(final_df['circuit_id'])

# -------------------------
# Step 3: Feature engineering
# -------------------------
final_df['driver_avg_finish_last5'] = final_df.groupby('driver')['position'] \
    .transform(lambda x: x.rolling(5, min_periods=1).mean())

final_df['driver_circuit_avg_finish'] = final_df.groupby(
    ['driver','circuit_id']
)['position'].transform('mean')

features = [
    'grid', 'driver_points', 'driver_wins', 'driver_standings_pos',
    'driver_avg_finish_last5', 'driver_circuit_avg_finish',
    'constructor_points', 'constructor_wins', 'constructor_standings_pos',
    'driver_enc', 'constructor_enc', 'circuit_enc'
]

final_df[features] = final_df[features].fillna(0)

# -------------------------
# Step 4: Train / Validate split
# -------------------------
train = final_df[final_df['season'] < 2025]
validate = final_df[final_df['season'] == 2025]

X_train = train[features]
y_train = train['top3']

X_validate = validate[features]
y_validate = validate['top3']

# -------------------------
# Step 5: Train Random Forest
# -------------------------
model = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

# Validate
y_pred_val = model.predict(X_validate)
print("2025 Validation Report:")
print(classification_report(y_validate, y_pred_val))

# -------------------------
# Step 6: Build 2026 dataset
# -------------------------
driver_stats_2025 = final_df[final_df['season'] == 2025].groupby('driver').agg({
    'driver_points':'sum',
    'driver_wins':'sum',
    'driver_standings_pos':'min',
    'driver_avg_finish_last5':'mean',
    'driver_circuit_avg_finish':'mean'
}).reset_index()

constructor_stats_2025 = final_df[final_df['season'] == 2025].groupby('constructor').agg({
    'constructor_points':'sum',
    'constructor_wins':'sum',
    'constructor_standings_pos':'min'
}).reset_index()

# ✅ FIX: use 2025 calendar
circuits_2026 = final_df[final_df['season']==2025]['circuit_id'].unique()
drivers_2025 = driver_stats_2025['driver'].unique()

rows_2026 = []

for round_, circuit in enumerate(circuits_2026, start=1):
    for driver in drivers_2025:

        constructor = final_df[
            (final_df['season']==2025) & 
            (final_df['driver']==driver)
        ]['constructor'].iloc[0]

        driver_row = driver_stats_2025[
            driver_stats_2025['driver']==driver
        ].iloc[0]

        constructor_row = constructor_stats_2025[
            constructor_stats_2025['constructor']==constructor
        ].iloc[0]

        rows_2026.append({
            'season':2026,
            'round':round_,
            'circuit_id':circuit,
            'driver':driver,
            'constructor':constructor,
            'grid':0,
            'driver_points':driver_row['driver_points'],
            'driver_wins':driver_row['driver_wins'],
            'driver_standings_pos':driver_row['driver_standings_pos'],
            'driver_avg_finish_last5':driver_row['driver_avg_finish_last5'],
            'driver_circuit_avg_finish':driver_row['driver_circuit_avg_finish'],
            'constructor_points':constructor_row['constructor_points'],
            'constructor_wins':constructor_row['constructor_wins'],
            'constructor_standings_pos':constructor_row['constructor_standings_pos'],
        })

df_2026_features = pd.DataFrame(rows_2026)

# -------------------------
# Step 7: Safe encoding
# -------------------------
def safe_transform(le, values):
    return [le.transform([v])[0] if v in le.classes_ else -1 for v in values]

df_2026_features['driver_enc'] = safe_transform(le_driver, df_2026_features['driver'])
df_2026_features['constructor_enc'] = safe_transform(le_constructor, df_2026_features['constructor'])
df_2026_features['circuit_enc'] = safe_transform(le_circuit, df_2026_features['circuit_id'])

# -------------------------
# Step 8: Predict
# -------------------------
X_2026 = df_2026_features[features]

df_2026_features['top3_prob'] = model.predict_proba(X_2026)[:,1]

predicted_podium_2026_RF = (
    df_2026_features
    .sort_values(['round','top3_prob'], ascending=[True, False])
    .groupby('round')
    .head(3)
)

print(predicted_podium_2026_RF[['round','driver','constructor','top3_prob']])

predicted_podium_2026_RF.to_csv("predicted_podium_rf_2026.csv", index=False)

2025 Validation Report:
              precision    recall  f1-score   support

           0       0.97      0.97      0.97       407
           1       0.81      0.81      0.81        72

    accuracy                           0.94       479
   macro avg       0.89      0.89      0.89       479
weighted avg       0.94      0.94      0.94       479

     round          driver constructor  top3_prob
13       1  max_verstappen    red_bull   0.629820
16       1         piastri     mclaren   0.521181
17       1         russell    mercedes   0.516355
34       2  max_verstappen    red_bull   0.623734
35       2          norris     mclaren   0.593017
..     ...             ...         ...        ...
478     23         piastri     mclaren   0.548347
476     23          norris     mclaren   0.540619
496     24  max_verstappen    red_bull   0.646082
500     24         russell    mercedes   0.616293
499     24         piastri     mclaren   0.615441

[72 rows x 4 columns]


In [80]:
#XGBClassifier
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from sklearn.metrics import classification_report

# -------------------------
# Step 1: Fill missing numeric values
# -------------------------
numeric_features = [
    'grid', 'driver_points', 'driver_wins', 'driver_standings_pos',
    'constructor_points', 'constructor_wins', 'constructor_standings_pos'
]

final_df[numeric_features] = final_df[numeric_features].fillna(0)

# -------------------------
# Step 2: Encode categorical variables
# -------------------------
final_df['driver'] = final_df['driver'].astype(str)
final_df['constructor'] = final_df['constructor'].astype(str)
final_df['circuit_id'] = final_df['circuit_id'].astype(str)

le_driver = LabelEncoder()
final_df['driver_enc'] = le_driver.fit_transform(final_df['driver'])

le_constructor = LabelEncoder()
final_df['constructor_enc'] = le_constructor.fit_transform(final_df['constructor'])

le_circuit = LabelEncoder()
final_df['circuit_enc'] = le_circuit.fit_transform(final_df['circuit_id'])

# -------------------------
# Step 3: Feature engineering
# -------------------------
final_df['driver_avg_finish_last5'] = final_df.groupby('driver')['position'] \
    .transform(lambda x: x.rolling(5, min_periods=1).mean())

final_df['driver_circuit_avg_finish'] = final_df.groupby(
    ['driver','circuit_id']
)['position'].transform('mean')

features = [
    'grid', 'driver_points', 'driver_wins', 'driver_standings_pos',
    'driver_avg_finish_last5', 'driver_circuit_avg_finish',
    'constructor_points', 'constructor_wins', 'constructor_standings_pos',
    'driver_enc', 'constructor_enc', 'circuit_enc'
]

final_df[features] = final_df[features].fillna(0)

# -------------------------
# Step 4: Train / Validate split
# -------------------------
train = final_df[final_df['season'] < 2025]
validate = final_df[final_df['season'] == 2025]

X_train = train[features]
y_train = train['top3']

X_validate = validate[features]
y_validate = validate['top3']

# -------------------------
# Step 5: Train XGBoost
# -------------------------
model = XGBClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='binary:logistic',
    eval_metric='logloss',
    use_label_encoder=False,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

# Validate
y_pred_val = model.predict(X_validate)
print("2025 Validation Report:")
print(classification_report(y_validate, y_pred_val))

# -------------------------
# Step 6: Prepare 2026 features
# -------------------------
driver_stats_2025 = final_df[final_df['season'] == 2025].groupby('driver').agg({
    'driver_points':'sum',
    'driver_wins':'sum',
    'driver_standings_pos':'min',
    'driver_avg_finish_last5':'mean',
    'driver_circuit_avg_finish':'mean'
}).reset_index()

constructor_stats_2025 = final_df[final_df['season'] == 2025].groupby('constructor').agg({
    'constructor_points':'sum',
    'constructor_wins':'sum',
    'constructor_standings_pos':'min'
}).reset_index()

circuits_2026 = final_df[final_df['season']==2025]['circuit_id'].unique()
drivers_2025 = driver_stats_2025['driver'].unique()

rows_2026 = []
for round_, circuit in enumerate(circuits_2026, start=1):
    for driver in drivers_2025:
        constructor = final_df[
            (final_df['season']==2025) & 
            (final_df['driver']==driver)
        ]['constructor'].iloc[0]

        driver_row = driver_stats_2025[
            driver_stats_2025['driver']==driver
        ].iloc[0]

        constructor_row = constructor_stats_2025[
            constructor_stats_2025['constructor']==constructor
        ].iloc[0]

        rows_2026.append({
            'season':2026,
            'round':round_,
            'circuit_id':circuit,
            'driver':driver,
            'constructor':constructor,
            'grid':0,
            'driver_points':driver_row['driver_points'],
            'driver_wins':driver_row['driver_wins'],
            'driver_standings_pos':driver_row['driver_standings_pos'],
            'driver_avg_finish_last5':driver_row['driver_avg_finish_last5'],
            'driver_circuit_avg_finish':driver_row['driver_circuit_avg_finish'],
            'constructor_points':constructor_row['constructor_points'],
            'constructor_wins':constructor_row['constructor_wins'],
            'constructor_standings_pos':constructor_row['constructor_standings_pos'],
        })

df_2026_features = pd.DataFrame(rows_2026)

# -------------------------
# Step 7: Safe encoding for 2026
# -------------------------
def safe_transform(le, values):
    return [le.transform([v])[0] if v in le.classes_ else -1 for v in values]

df_2026_features['driver_enc'] = safe_transform(le_driver, df_2026_features['driver'])
df_2026_features['constructor_enc'] = safe_transform(le_constructor, df_2026_features['constructor'])
df_2026_features['circuit_enc'] = safe_transform(le_circuit, df_2026_features['circuit_id'])

# -------------------------
# Step 8: Predict 2026 top-3
# -------------------------
X_2026 = df_2026_features[features]

df_2026_features['top3_prob'] = model.predict_proba(X_2026)[:,1]

predicted_podium_2026_XGB = (
    df_2026_features
    .sort_values(['round','top3_prob'], ascending=[True, False])
    .groupby('round')
    .head(3)
)

print(predicted_podium_2026_XGB[['round','driver','constructor','top3_prob']])

predicted_podium_2026_XGB.to_csv("predicted_podium_xgb_2026.csv", index=False)

C:\Users\RObus\anaconda3\envs\survival_env\lib\site-packages\xgboost\training.py:200: UserWarning: [15:31:19] WARNING: C:\Users\task_177465309458303\croot\xgboost-split_1774653200626\work\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


2025 Validation Report:
              precision    recall  f1-score   support

           0       0.97      0.97      0.97       407
           1       0.82      0.83      0.83        72

    accuracy                           0.95       479
   macro avg       0.90      0.90      0.90       479
weighted avg       0.95      0.95      0.95       479

     round          driver constructor  top3_prob
13       1  max_verstappen    red_bull   0.474261
12       1         leclerc     ferrari   0.427665
2        1       antonelli    mercedes   0.308758
34       2  max_verstappen    red_bull   0.656079
33       2         leclerc     ferrari   0.620577
..     ...             ...         ...        ...
474     23         leclerc     ferrari   0.524554
479     23         russell    mercedes   0.359350
496     24  max_verstappen    red_bull   0.729585
495     24         leclerc     ferrari   0.709635
497     24          norris     mclaren   0.632225

[72 rows x 4 columns]


In [81]:
#LGBMClassifier
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from lightgbm import LGBMClassifier
from sklearn.metrics import classification_report

# -------------------------
# Step 1: Fill missing numeric values
# -------------------------
numeric_features = [
    'grid', 'driver_points', 'driver_wins', 'driver_standings_pos',
    'constructor_points', 'constructor_wins', 'constructor_standings_pos'
]

final_df[numeric_features] = final_df[numeric_features].fillna(0)

# -------------------------
# Step 2: Encode categorical variables
# -------------------------
final_df['driver'] = final_df['driver'].astype(str)
final_df['constructor'] = final_df['constructor'].astype(str)
final_df['circuit_id'] = final_df['circuit_id'].astype(str)

le_driver = LabelEncoder()
final_df['driver_enc'] = le_driver.fit_transform(final_df['driver'])

le_constructor = LabelEncoder()
final_df['constructor_enc'] = le_constructor.fit_transform(final_df['constructor'])

le_circuit = LabelEncoder()
final_df['circuit_enc'] = le_circuit.fit_transform(final_df['circuit_id'])

# -------------------------
# Step 3: Feature engineering
# -------------------------
final_df['driver_avg_finish_last5'] = final_df.groupby('driver')['position'] \
    .transform(lambda x: x.rolling(5, min_periods=1).mean())

final_df['driver_circuit_avg_finish'] = final_df.groupby(
    ['driver','circuit_id']
)['position'].transform('mean')

features = [
    'grid', 'driver_points', 'driver_wins', 'driver_standings_pos',
    'driver_avg_finish_last5', 'driver_circuit_avg_finish',
    'constructor_points', 'constructor_wins', 'constructor_standings_pos',
    'driver_enc', 'constructor_enc', 'circuit_enc'
]

final_df[features] = final_df[features].fillna(0)

# -------------------------
# Step 4: Train / Validate split
# -------------------------
train = final_df[final_df['season'] < 2025]
validate = final_df[final_df['season'] == 2025]

X_train = train[features]
y_train = train['top3']

X_validate = validate[features]
y_validate = validate['top3']

# -------------------------
# Step 5: Train LightGBM
# -------------------------
model = LGBMClassifier(
    n_estimators=300,
    num_leaves=31,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='binary',
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

# Validate
y_pred_val = model.predict(X_validate)
print("2025 Validation Report:")
print(classification_report(y_validate, y_pred_val))

# -------------------------
# Step 6: Prepare 2026 features
# -------------------------
driver_stats_2025 = final_df[final_df['season'] == 2025].groupby('driver').agg({
    'driver_points':'sum',
    'driver_wins':'sum',
    'driver_standings_pos':'min',
    'driver_avg_finish_last5':'mean',
    'driver_circuit_avg_finish':'mean'
}).reset_index()

constructor_stats_2025 = final_df[final_df['season'] == 2025].groupby('constructor').agg({
    'constructor_points':'sum',
    'constructor_wins':'sum',
    'constructor_standings_pos':'min'
}).reset_index()

circuits_2026 = final_df[final_df['season']==2025]['circuit_id'].unique()
drivers_2025 = driver_stats_2025['driver'].unique()

rows_2026 = []
for round_, circuit in enumerate(circuits_2026, start=1):
    for driver in drivers_2025:
        constructor = final_df[
            (final_df['season']==2025) & 
            (final_df['driver']==driver)
        ]['constructor'].iloc[0]

        driver_row = driver_stats_2025[
            driver_stats_2025['driver']==driver
        ].iloc[0]

        constructor_row = constructor_stats_2025[
            constructor_stats_2025['constructor']==constructor
        ].iloc[0]

        rows_2026.append({
            'season':2026,
            'round':round_,
            'circuit_id':circuit,
            'driver':driver,
            'constructor':constructor,
            'grid':0,
            'driver_points':driver_row['driver_points'],
            'driver_wins':driver_row['driver_wins'],
            'driver_standings_pos':driver_row['driver_standings_pos'],
            'driver_avg_finish_last5':driver_row['driver_avg_finish_last5'],
            'driver_circuit_avg_finish':driver_row['driver_circuit_avg_finish'],
            'constructor_points':constructor_row['constructor_points'],
            'constructor_wins':constructor_row['constructor_wins'],
            'constructor_standings_pos':constructor_row['constructor_standings_pos'],
        })

df_2026_features = pd.DataFrame(rows_2026)

# -------------------------
# Step 7: Safe encoding for 2026
# -------------------------
def safe_transform(le, values):
    return [le.transform([v])[0] if v in le.classes_ else -1 for v in values]

df_2026_features['driver_enc'] = safe_transform(le_driver, df_2026_features['driver'])
df_2026_features['constructor_enc'] = safe_transform(le_constructor, df_2026_features['constructor'])
df_2026_features['circuit_enc'] = safe_transform(le_circuit, df_2026_features['circuit_id'])

# -------------------------
# Step 8: Predict 2026 top-3
# -------------------------
X_2026 = df_2026_features[features]
df_2026_features['top3_prob'] = model.predict_proba(X_2026)[:,1]

predicted_podium_2026_gbm = (
    df_2026_features
    .sort_values(['round','top3_prob'], ascending=[True, False])
    .groupby('round')
    .head(3)
)

print(predicted_podium_2026_gbm[['round','driver','constructor','top3_prob']])

predicted_podium_2026_gbm.to_csv("predicted_podium_lgb_2026.csv", index=False, encoding="utf-8-sig")

[LightGBM] [Info] Number of positive: 270, number of negative: 1546
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000592 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 931
[LightGBM] [Info] Number of data points in the train set: 1816, number of used features: 12
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.148678 -> initscore=-1.745004
[LightGBM] [Info] Start training from score -1.745004
2025 Validation Report:
              precision    recall  f1-score   support

           0       0.97      0.97      0.97       407
           1       0.81      0.82      0.81        72

    accuracy                           0.94       479
   macro avg       0.89      0.89      0.89       479
weighted avg       0.94      0.94      0.94       479

     round          driver   constructor  top3_prob
12       1         leclerc       ferrari   0

In [82]:
#XGB + Weather and Qual
import pandas as pd 
#from sklearn.preprocessing import LabelEncoder 
#from sklearn.ensemble import HistGradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
# Step 1: Fill missing numeric values for columns that exist 
#numeric_features = [ 'grid', 'driver_points', 'driver_wins', 'driver_standings_pos', 'constructor_points', 'constructor_wins',
                     #'constructor_standings_pos', 'weather_warm', 'weather_cold', 'weather_dry', 'weather_wet', 'weather_cloudy' ]

# -------------------------
# Qualifying Features
# -------------------------

# Convert qualifying time to numeric seconds if needed
final_df['qualifying_time'] = pd.to_numeric(
    final_df['qualifying_time'],
    errors='coerce'
)

# Pole time for each race
final_df['pole_time'] = (
    final_df.groupby(['season', 'round'])['qualifying_time']
    .transform('min')
)

# Gap to pole
final_df['qualifying_gap'] = (
    final_df['qualifying_time']
    - final_df['pole_time']
)

# Driver qualifying form (previous 5 races)
final_df['avg_qualifying_gap_last5'] = (
    final_df.groupby('driver')['qualifying_gap']
    .transform(
        lambda x: x.shift(1)
                  .rolling(5, min_periods=1)
                  .mean()
    )
)






numeric_features = [
    'grid',
    'qualifying_time',
    'qualifying_gap',
    'avg_qualifying_gap_last5',
    'driver_points',
    'driver_wins',
    'driver_standings_pos',
    'constructor_points',
    'constructor_wins',
    'constructor_standings_pos',
    'weather_warm',
    'weather_cold',
    'weather_dry',
    'weather_wet',
    'weather_cloudy'
]
# Fill missing values for numeric features, including weather 
final_df[numeric_features] = final_df[numeric_features].fillna(0) 
# Step 2: Encode categorical variables 
le_driver = LabelEncoder()
final_df['driver_enc'] = le_driver.fit_transform(final_df['driver']) 
le_constructor = LabelEncoder() 
final_df['constructor_enc'] = le_constructor.fit_transform(final_df['constructor']) 
le_circuit = LabelEncoder() 
final_df['circuit_enc'] = le_circuit.fit_transform(final_df['circuit_id']) 
# Step 3: Compute rolling / historical features (e.g., last 5 races, circuit avg) 
final_df['driver_avg_finish_last5'] = final_df.groupby('driver')['position'].transform(lambda x: x.rolling(5, min_periods=1).mean())
final_df['driver_circuit_avg_finish'] = final_df.groupby(['driver', 'circuit_id'])['position'].transform('mean') 





# Step 4: Prepare feature set with weather features included 
#features = [ 'grid', 'driver_points', 'driver_wins', 'driver_standings_pos',
#'driver_avg_finish_last5', 'driver_circuit_avg_finish', 'constructor_points', 'constructor_wins', 'constructor_standings_pos',
             #'driver_enc', 'constructor_enc', 'circuit_enc', 'weather_warm', 'weather_cold', 'weather_dry', 'weather_wet', 'weather_cloudy' ]

features = [
    'grid',

    'qualifying_time',
    'qualifying_gap',
    'avg_qualifying_gap_last5',

    'driver_points',
    'driver_wins',
    'driver_standings_pos',

    'driver_avg_finish_last5',
    'driver_circuit_avg_finish',

    'constructor_points',
    'constructor_wins',
    'constructor_standings_pos',

    'driver_enc',
    'constructor_enc',
    'circuit_enc',

    'weather_warm',
    'weather_cold',
    'weather_dry',
    'weather_wet',
    'weather_cloudy'
]
# Fill any remaining NaNs (from rolling averages and weather features) 
final_df[features] = final_df[features].fillna(0)
# ------------------------- # 1️⃣ Prepare features & target # ------------------------- 
features = [ 'grid', 'driver_points', 'driver_wins', 'driver_standings_pos', 'driver_avg_finish_last5', 'driver_circuit_avg_finish',
             'constructor_points', 'constructor_wins', 'constructor_standings_pos', 'driver_enc', 'constructor_enc', 'circuit_enc',
             'weather_warm', 'weather_cold', 'weather_dry', 'weather_wet', 'weather_cloudy' ]
# Fill missing numeric values in historical data 
final_df[features] = final_df[features].fillna(0) 
# ------------------------- # 2️⃣ Encode drivers, constructors, circuits # ------------------------- 
final_df['driver'] = final_df['driver'].apply(str) 
final_df['constructor'] = final_df['constructor'].apply(str)
final_df['circuit'] = final_df['circuit_id'].apply(str)
le_driver = LabelEncoder() 
final_df['driver_enc'] = le_driver.fit_transform(final_df['driver']) 
le_constructor = LabelEncoder()
final_df['constructor_enc'] = le_constructor.fit_transform(final_df['constructor']) 
le_circuit = LabelEncoder() 
final_df['circuit_enc'] = le_circuit.fit_transform(final_df['circuit_id']) 
# ------------------------- # 3️⃣ Split training / validation # ------------------------- 
train = final_df[final_df['season'] < 2025] 
# 2021–2024 
validate = final_df[final_df['season'] == 2025] 
# 2025 
X_train = train[features] 
y_train = train['top3']
X_validate = validate[features] 
y_validate = validate['top3'] 

# -------------------------
# 4️⃣ Train XGBoost Model
# -------------------------

# Calculate class imbalance ratio
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()

model = XGBClassifier(
    n_estimators=1000,
    learning_rate=0.03,
    max_depth=8,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    objective='binary:logistic',
    eval_metric='logloss',
    scale_pos_weight=neg / pos,
    random_state=42
)

model.fit(X_train, y_train)

# -------------------------
# Validation
# -------------------------

y_pred_val = model.predict(X_validate)

print("2025 Validation Report:")
print(classification_report(y_validate, y_pred_val))

# 5️⃣ Prepare 2026 features from 2025 stats
# -------------------------

driver_stats_2025 = (
    final_df[final_df['season'] == 2025]
    .groupby('driver')
    .agg({
        'driver_points': 'sum',
        'driver_wins': 'sum',
        'driver_standings_pos': 'min',
        'driver_avg_finish_last5': 'mean',
        'driver_circuit_avg_finish': 'mean',
        'qualifying_time': 'mean',
        'qualifying_gap': 'mean',
        'avg_qualifying_gap_last5': 'mean'
    })
    .reset_index()
)



constructor_stats_2025 = (
    final_df[final_df['season'] == 2025]
    .groupby('constructor')
    .agg({
        'constructor_points': 'sum',
        'constructor_wins': 'sum',
        'constructor_standings_pos': 'min'
    })
    .reset_index()
)

circuits_2026 = final_df.loc[
    final_df['season'] == 2026,
    'circuit_id'
].unique()

drivers_2025 = driver_stats_2025['driver'].unique()

rows_2026 = []

for round_, circuit in enumerate(circuits_2026, start=1):

    # Get weather for this circuit from historical races
    subset = final_df[
        (final_df['season'] == 2025) &
        (final_df['circuit_id'] == circuit)
    ]

    if subset.empty:
        weather_values = {
            'weather_warm': 0,
            'weather_cold': 0,
            'weather_dry': 1,
            'weather_wet': 0,
            'weather_cloudy': 0
        }
    else:
        weather_row = subset.iloc[0]

        weather_values = {
            'weather_warm': weather_row['weather_warm'],
            'weather_cold': weather_row['weather_cold'],
            'weather_dry': weather_row['weather_dry'],
            'weather_wet': weather_row['weather_wet'],
            'weather_cloudy': weather_row['weather_cloudy']
        }

    # LOOP OVER ALL DRIVERS
    for driver in drivers_2025:

        constructor = (
            final_df[
                (final_df['season'] == 2025) &
                (final_df['driver'] == driver)
            ]['constructor']
            .iloc[0]
        )

        driver_row = (
            driver_stats_2025[
                driver_stats_2025['driver'] == driver
            ]
            .iloc[0]
        )

        constructor_row = (
            constructor_stats_2025[
                constructor_stats_2025['constructor'] == constructor
            ]
            .iloc[0]
        )

        rows_2026.append({
            'season': 2026,
            'round': round_,
            'circuit_id': circuit,

            'driver': driver,
            'constructor': constructor,

            # placeholder qualifying position
            'grid': 10,

            'driver_points': driver_row['driver_points'],
            'driver_wins': driver_row['driver_wins'],
            'driver_standings_pos': driver_row['driver_standings_pos'],
            'driver_avg_finish_last5': driver_row['driver_avg_finish_last5'],
            'driver_circuit_avg_finish': driver_row['driver_circuit_avg_finish'],

            'constructor_points': constructor_row['constructor_points'],
            'constructor_wins': constructor_row['constructor_wins'],
            'constructor_standings_pos': constructor_row['constructor_standings_pos'],

            'qualifying_time': driver_row['qualifying_time'],
            'qualifying_gap': driver_row['qualifying_gap'],
            'avg_qualifying_gap_last5': driver_row['avg_qualifying_gap_last5'],

            **weather_values
        })

df_2026_features = pd.DataFrame(rows_2026)

print("Rows:", len(df_2026_features))
print("Unique drivers:", df_2026_features['driver'].nunique())

# Encode drivers / constructors / circuits for 2026 
df_2026_features['driver_enc'] = le_driver.transform(df_2026_features['driver']) 
df_2026_features['constructor_enc'] = le_constructor.transform(df_2026_features['constructor'])
df_2026_features['circuit_enc'] = le_circuit.transform(df_2026_features['circuit_id']) 
# -------------------------
# 6️⃣ Predict Top-3 probabilities for 2026
# -------------------------

X_2026 = df_2026_features[features]

df_2026_features['top3_prob'] = (
    model.predict_proba(X_2026)[:, 1]
)

predicted_podium_2026 = (
    df_2026_features
    .sort_values(
        ['round', 'top3_prob'],
        ascending=[True, False]
    )
    .groupby('round')
    .head(3)
)

print(
    predicted_podium_2026[
        ['round', 'driver', 'constructor', 'top3_prob']
    ]
)

predicted_podium_2026.to_csv(
    "predicted_podium_2026_XGB_W.csv",
    index=False,
    encoding="utf-8-sig"
)




2025 Validation Report:
              precision    recall  f1-score   support

           0       0.98      0.95      0.96       407
           1       0.75      0.89      0.82        72

    accuracy                           0.94       479
   macro avg       0.87      0.92      0.89       479
weighted avg       0.95      0.94      0.94       479

Rows: 483
Unique drivers: 21
     round          driver constructor  top3_prob
13       1  max_verstappen    red_bull   0.176993
9        1        hamilton     ferrari   0.111706
17       1         russell    mercedes   0.106274
30       2        hamilton     ferrari   0.288936
33       2         leclerc     ferrari   0.157039
..     ...             ...         ...        ...
450     22        hamilton     ferrari   0.180714
457     22         piastri     mclaren   0.139373
471     23        hamilton     ferrari   0.601456
474     23         leclerc     ferrari   0.398196
478     23         piastri     mclaren   0.326185

[69 rows x 4 column

In [83]:
#histgradientWQRolling2026
# ============================================================
# 1. LOAD & PREPARE DATA
# ============================================================

import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import classification_report

# ------------------------------------------------------------
# Copy dataframe
# ------------------------------------------------------------

df = final_df.copy()

# ------------------------------------------------------------
# Sort chronologically
# ------------------------------------------------------------

df = df.sort_values(
    ["season", "round", "grid"]
).reset_index(drop=True)

# ------------------------------------------------------------
# Create race date (if available)
# ------------------------------------------------------------

if "date" in df.columns:
    df["date"] = pd.to_datetime(df["date"])
else:
    # fallback ordering
    df["date"] = pd.to_datetime(
        df["season"].astype(str)
    ) + pd.to_timedelta(df["round"], unit="D")

# ------------------------------------------------------------
# Convert numeric columns
# ------------------------------------------------------------

numeric_cols = [

    "grid",
    "position",

    "driver_points",
    "driver_wins",
    "driver_standings_pos",

    "constructor_points",
    "constructor_wins",
    "constructor_standings_pos",

    "qualifying_time"

]

for col in numeric_cols:

    if col in df.columns:
        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )

# ------------------------------------------------------------
# Fill weather columns
# ------------------------------------------------------------

weather_cols = [

    "weather_warm",
    "weather_cold",
    "weather_dry",
    "weather_wet",
    "weather_cloudy"

]

for col in weather_cols:

    if col not in df.columns:
        df[col] = 0

df[weather_cols] = df[weather_cols].fillna(0)

# ------------------------------------------------------------
# Create target variable
# ------------------------------------------------------------

df["top3"] = (
    df["position"] <= 3
).astype(int)

# ------------------------------------------------------------
# Standardise text columns
# ------------------------------------------------------------

df["driver"] = df["driver"].astype(str)
df["constructor"] = df["constructor"].astype(str)
df["circuit_id"] = df["circuit_id"].astype(str)

# ------------------------------------------------------------
# Label encoders
# ------------------------------------------------------------

driver_encoder = LabelEncoder()
constructor_encoder = LabelEncoder()
circuit_encoder = LabelEncoder()

df["driver_enc"] = driver_encoder.fit_transform(df["driver"])

df["constructor_enc"] = constructor_encoder.fit_transform(
    df["constructor"]
)

df["circuit_enc"] = circuit_encoder.fit_transform(
    df["circuit_id"]
)

# ------------------------------------------------------------
# Fill remaining numeric NaNs
# ------------------------------------------------------------

df = df.sort_values(
    ["driver", "season", "round"]
)

numeric = df.select_dtypes(include=np.number).columns

df[numeric] = df[numeric].fillna(0)

# ------------------------------------------------------------
# Final ordering
# ------------------------------------------------------------

df = df.sort_values(
    ["season", "round", "grid"]
).reset_index(drop=True)

print(df.shape)
print(df.head())
# ============================================================
# 2. FEATURE ENGINEERING (LEAKAGE FREE)
# ============================================================

# ------------------------------------------------------------
# Sort chronologically
# ------------------------------------------------------------

df = df.sort_values(
    ["driver", "season", "round"]
).reset_index(drop=True)

# ------------------------------------------------------------
# Qualifying Features
# ------------------------------------------------------------

# Pole time for each race
df["pole_time"] = (
    df.groupby(["season", "round"])["qualifying_time"]
      .transform("min")
)

# Gap to pole
df["qualifying_gap"] = (
    df["qualifying_time"] - df["pole_time"]
)

# Previous 5 qualifying performances
df["avg_qualifying_gap_last5"] = (
    df.groupby("driver")["qualifying_gap"]
      .transform(
          lambda x:
              x.shift(1)
               .rolling(5, min_periods=1)
               .mean()
      )
)

# ------------------------------------------------------------
# Driver recent form
# ------------------------------------------------------------

df["driver_avg_finish_last5"] = (
    df.groupby("driver")["position"]
      .transform(
          lambda x:
              x.shift(1)
               .rolling(5, min_periods=1)
               .mean()
      )
)

# Best finish in previous 5 races
df["driver_best_finish_last5"] = (
    df.groupby("driver")["position"]
      .transform(
          lambda x:
              x.shift(1)
               .rolling(5, min_periods=1)
               .min()
      )
)

# ------------------------------------------------------------
# Driver podium rate
# ------------------------------------------------------------

df["previous_top3"] = (
    (df["position"] <= 3)
    .astype(int)
)

df["driver_top3_rate_last5"] = (
    df.groupby("driver")["previous_top3"]
      .transform(
          lambda x:
              x.shift(1)
               .rolling(5, min_periods=1)
               .mean()
      )
)

# ------------------------------------------------------------
# Driver average at circuit
# Previous visits ONLY
# ------------------------------------------------------------

df["driver_circuit_avg_finish"] = np.nan

for (driver, circuit), idx in df.groupby(
    ["driver", "circuit_id"]
).groups.items():

    history = []

    for i in idx:

        if len(history) == 0:
            df.loc[i, "driver_circuit_avg_finish"] = np.nan
        else:
            df.loc[i, "driver_circuit_avg_finish"] = np.mean(history)

        history.append(df.loc[i, "position"])

# ------------------------------------------------------------
# Constructor recent form
# ------------------------------------------------------------

df["constructor_avg_finish_last5"] = (
    df.groupby("constructor")["position"]
      .transform(
          lambda x:
              x.shift(1)
               .rolling(5, min_periods=1)
               .mean()
      )
)

# ------------------------------------------------------------
# Previous grid position
# ------------------------------------------------------------

df["avg_grid_last5"] = (
    df.groupby("driver")["grid"]
      .transform(
          lambda x:
              x.shift(1)
               .rolling(5, min_periods=1)
               .mean()
      )
)

# ------------------------------------------------------------
# Fill NaNs
# ------------------------------------------------------------

rolling_features = [

    "qualifying_gap",
    "avg_qualifying_gap_last5",

    "driver_avg_finish_last5",
    "driver_best_finish_last5",
    "driver_top3_rate_last5",

    "driver_circuit_avg_finish",

    "constructor_avg_finish_last5",

    "avg_grid_last5"

]

df[rolling_features] = (
    df[rolling_features]
    .fillna(0)
)

print("Feature engineering complete.")
print(df[rolling_features].head())
# ============================================================
# 3. LEAKAGE-FREE ROLLING STATISTICS
# ============================================================

# Make sure races are chronological
df = df.sort_values(
    ["season", "round", "position"]
).reset_index(drop=True)

# ------------------------------------------------------------
# Initialise columns
# ------------------------------------------------------------

cols = [
    "driver_points",
    "driver_wins",
    "driver_standings_pos",
    "constructor_points",
    "constructor_wins",
    "constructor_standings_pos"
]

for c in cols:
    df[c] = 0

# FIA points system
POINTS = {
    1:25,
    2:18,
    3:15,
    4:12,
    5:10,
    6:8,
    7:6,
    8:4,
    9:2,
    10:1
}

# Running totals
driver_points = {}
constructor_points = {}

driver_wins = {}
constructor_wins = {}

# ------------------------------------------------------------
# Loop race by race
# ------------------------------------------------------------

for (season, rnd), race in df.groupby(
    ["season","round"],
    sort=True
):

    # ---------- standings BEFORE this race ----------

    driver_rank = (
        pd.Series(driver_points)
        .sort_values(ascending=False)
    )

    constructor_rank = (
        pd.Series(constructor_points)
        .sort_values(ascending=False)
    )

    driver_pos = {
        d:i+1
        for i,d in enumerate(driver_rank.index)
    }

    constructor_pos = {
        c:i+1
        for i,c in enumerate(constructor_rank.index)
    }

    # Save values BEFORE race
    for idx,row in race.iterrows():

        d = row["driver"]
        c = row["constructor"]

        df.loc[idx,"driver_points"] = driver_points.get(d,0)
        df.loc[idx,"driver_wins"] = driver_wins.get(d,0)

        df.loc[idx,"constructor_points"] = constructor_points.get(c,0)
        df.loc[idx,"constructor_wins"] = constructor_wins.get(c,0)

        df.loc[idx,"driver_standings_pos"] = driver_pos.get(
            d,
            len(driver_pos)+1
        )

        df.loc[idx,"constructor_standings_pos"] = constructor_pos.get(
            c,
            len(constructor_pos)+1
        )

    # ---------- update after race ----------

    for idx,row in race.iterrows():

        d = row["driver"]
        c = row["constructor"]

        pts = POINTS.get(
            int(row["position"]),
            0
        )

        driver_points[d] = driver_points.get(d,0) + pts
        constructor_points[c] = constructor_points.get(c,0) + pts

        if row["position"] == 1:

            driver_wins[d] = driver_wins.get(d,0) + 1
            constructor_wins[c] = constructor_wins.get(c,0) + 1

print("Leakage-free standings created.")
# ============================================================
# 4. WALK-FORWARD TRAINING & VALIDATION
# ============================================================

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import classification_report

# ------------------------------------------------------------
# Features
# ------------------------------------------------------------

features = [

    "grid",

    "qualifying_time",
    "qualifying_gap",
    "avg_qualifying_gap_last5",

    "driver_points",
    "driver_wins",
    "driver_standings_pos",

    "driver_avg_finish_last5",
    "driver_best_finish_last5",
    "driver_top3_rate_last5",
    "driver_circuit_avg_finish",

    "constructor_points",
    "constructor_wins",
    "constructor_standings_pos",
    "constructor_avg_finish_last5",

    "driver_enc",
    "constructor_enc",
    "circuit_enc",

    "weather_warm",
    "weather_cold",
    "weather_dry",
    "weather_wet",
    "weather_cloudy"
]

# Fill remaining NaNs
df[features] = df[features].fillna(0)

# ------------------------------------------------------------
# Predictions
# ------------------------------------------------------------

validation_predictions = []

# Validate on all races from 2025 onwards
validation_races = df[
    df["season"] >= 2025
][["season","round"]].drop_duplicates()

validation_races = validation_races.sort_values(
    ["season","round"]
)

# ------------------------------------------------------------
# Walk-forward loop
# ------------------------------------------------------------

for _, race_info in validation_races.iterrows():

    season = race_info["season"]
    rnd = race_info["round"]

    # -----------------------
    # Training data
    # -----------------------

    train = df[
        (df["season"] < season) |
        (
            (df["season"] == season) &
            (df["round"] < rnd)
        )
    ]

    # Skip first race if no history
    if len(train) < 200:
        continue

    # -----------------------
    # Race to predict
    # -----------------------

    test = df[
        (df["season"] == season) &
        (df["round"] == rnd)
    ].copy()

    X_train = train[features]
    y_train = train["top3"]

    X_test = test[features]

    # -----------------------
    # Train model
    # -----------------------

    model = HistGradientBoostingClassifier(

        max_iter=500,
        learning_rate=0.05,
        max_depth=6,
        random_state=42

    )

    model.fit(X_train, y_train)

    # -----------------------
    # Predict
    # -----------------------

    test["top3_prob"] = model.predict_proba(X_test)[:,1]
    test["prediction"] = (
        test["top3_prob"] >= 0.50
    ).astype(int)

    validation_predictions.append(test)

# ------------------------------------------------------------
# Combine predictions
# ------------------------------------------------------------

validation_results = pd.concat(
    validation_predictions,
    ignore_index=True
)

print(classification_report(
    validation_results["top3"],
    validation_results["prediction"]
))
# ============================================================
# 5. 2025 WALK-FORWARD VALIDATION
# ============================================================

from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score
)

# ------------------------------------------------------------
# Filter only 2025 predictions
# ------------------------------------------------------------

results_2025 = validation_results[
    validation_results["season"] == 2025
].copy()


# ------------------------------------------------------------
# Classification metrics
# ------------------------------------------------------------

print("="*60)
print("2025 WALK-FORWARD VALIDATION")
print("="*60)


print(
    classification_report(
        results_2025["top3"],
        results_2025["prediction"],
        digits=3
    )
)


# ------------------------------------------------------------
# Probability metrics
# ------------------------------------------------------------

auc = roc_auc_score(
    results_2025["top3"],
    results_2025["top3_prob"]
)

print(
    f"ROC-AUC: {auc:.3f}"
)


# ------------------------------------------------------------
# Podium prediction accuracy
# ------------------------------------------------------------

accuracy = accuracy_score(
    results_2025["top3"],
    results_2025["prediction"]
)

precision = precision_score(
    results_2025["top3"],
    results_2025["prediction"]
)

recall = recall_score(
    results_2025["top3"],
    results_2025["prediction"]
)


print(
    f"Accuracy : {accuracy:.3f}"
)

print(
    f"Precision: {precision:.3f}"
)

print(
    f"Recall   : {recall:.3f}"
)


# ------------------------------------------------------------
# Race-by-race podium predictions
# ------------------------------------------------------------

for rnd in sorted(
    results_2025["round"].unique()
):

    race = results_2025[
        results_2025["round"] == rnd
    ]

    print("\n")
    print(
        f"2025 Round {rnd}"
    )

    print(
        race
        .sort_values(
            "top3_prob",
            ascending=False
        )
        [
            [
                "driver",
                "constructor",
                "top3_prob",
                "top3"
            ]
        ]
        .head(5)
    )
    # ============================================================
# 6. 2026 ROLLING VALIDATION
# ============================================================

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import classification_report, roc_auc_score


# ------------------------------------------------------------
# Select completed 2026 races
# ------------------------------------------------------------

races_2026 = (
    df[df["season"] == 2026]
    [["season","round"]]
    .drop_duplicates()
    .sort_values("round")
)


rolling_predictions_2026 = []


# ------------------------------------------------------------
# Walk forward through 2026
# ------------------------------------------------------------

for _, race_info in races_2026.iterrows():

    rnd = race_info["round"]

    print(
        f"Predicting 2026 Round {rnd}"
    )


    # ----------------------------------------
    # Training data
    # ----------------------------------------

    train = df[
        (df["season"] < 2026) |
        (
            (df["season"] == 2026) &
            (df["round"] < rnd)
        )
    ]


    # ----------------------------------------
    # Current race
    # ----------------------------------------

    test = df[
        (df["season"] == 2026) &
        (df["round"] == rnd)
    ].copy()


    # Need previous races
    if len(train) == 0:
        continue


    X_train = train[features]
    y_train = train["top3"]

    X_test = test[features]


    # ----------------------------------------
    # Train fresh model
    # ----------------------------------------

    model_2026 = HistGradientBoostingClassifier(

        max_iter=500,
        learning_rate=0.05,
        max_depth=6,
        random_state=42

    )


    model_2026.fit(
        X_train,
        y_train
    )


    # ----------------------------------------
    # Predict
    # ----------------------------------------

    test["top3_prob"] = (
        model_2026
        .predict_proba(X_test)[:,1]
    )


    test["prediction"] = (
        test["top3_prob"] >= 0.50
    ).astype(int)


    rolling_predictions_2026.append(test)



# ------------------------------------------------------------
# Combine all 2026 predictions
# ------------------------------------------------------------

results_2026 = pd.concat(
    rolling_predictions_2026,
    ignore_index=True
)


print(
    "Completed 2026 races:",
    results_2026["round"].nunique()
)
# ============================================================
# 2026 VALIDATION METRICS
# ============================================================

print("="*60)
print("2026 ROLLING VALIDATION")
print("="*60)


print(
    classification_report(
        results_2026["top3"],
        results_2026["prediction"],
        digits=3
    )
)


auc_2026 = roc_auc_score(
    results_2026["top3"],
    results_2026["top3_prob"]
)


print(
    f"ROC-AUC: {auc_2026:.3f}"
)
# ============================================================
# 7. REMAINING 2026 PREDICTIONS + CSV OUTPUT
# ============================================================

import pandas as pd


# ------------------------------------------------------------
# Find latest completed 2026 round
# ------------------------------------------------------------

# ============================================================
# Find REAL completed 2026 races
# ============================================================

completed_2026 = df[
    (df["season"] == 2026) &
    (df["driver"] != "0") &
    (df["constructor"] != "0") &
    (df["position"] < 100)
]


race_counts = (
    completed_2026
    .groupby("round")["driver"]
    .count()
)


completed_rounds = (
    race_counts[
        race_counts >= 15
    ]
    .index
)


completed_2026 = completed_2026[
    completed_2026["round"].isin(completed_rounds)
]


latest_round = completed_2026["round"].max()


print("Completed rounds:")
print(completed_rounds.tolist())

print(
    "Latest completed round:",
    latest_round
)

# ------------------------------------------------------------
# Train final model
# ------------------------------------------------------------

training_final = df[
    (df["season"] < 2026) |
    (
        (df["season"] == 2026) &
        (df["round"].isin(completed_rounds))
    )
]


final_model = HistGradientBoostingClassifier(

    max_iter=500,
    learning_rate=0.05,
    max_depth=6,
    random_state=42

)


final_model.fit(
    training_final[features],
    training_final["top3"]
)


# ------------------------------------------------------------
# Future 2026 races
# ------------------------------------------------------------

# ============================================================
# Remaining 2026 calendar
# ============================================================

future_races = (
    df[
        (df["season"] == 2026) &
        (df["round"] > latest_round)
    ]
    [
        [
            "round",
            "circuit_id"
        ]
    ]
    .drop_duplicates()
)


print(future_races)


# ------------------------------------------------------------
# Create prediction dataframe
# ------------------------------------------------------------

future_rows = []


drivers = (
    completed_2026
    .loc[
        completed_2026["round"] == latest_round,
        "driver"
    ]
    .unique()
)

print(len(drivers))


for _, race in future_races.iterrows():

    rnd = race["round"]
    circuit = race["circuit_id"]


    # latest driver information

    latest_driver = (
        df[
            (
                (df["season"] < 2026) |
                (
                    (df["season"] == 2026) &
                    (df["round"] <= latest_round)
                )
            )
            &
            (df["circuit_id"] == circuit)
        ]
    )


    for driver in drivers:


        driver_history = df[
    (df["driver"] == driver) &
    (
        (df["season"] < 2026) |
        (
            (df["season"] == 2026) &
            (df["round"] <= latest_round)
        )
    )
]
        if driver_history.empty:
            continue

        if len(driver_history) == 0:
            continue


        last = (
            driver_history
            .sort_values(
                ["season","round"]
            )
            .iloc[-1]
        )


        constructor = last["constructor"]


        future_rows.append({

            "season":2026,
            "round":rnd,

            "driver":driver,
            "constructor":constructor,
            "circuit_id":circuit,


            # Current form

            "grid":
                last["grid"],

            "qualifying_time":
                last["qualifying_time"],

            "qualifying_gap":
                last["qualifying_gap"],

            "avg_qualifying_gap_last5":
                last["avg_qualifying_gap_last5"],


            "driver_points":
                last["driver_points"],

            "driver_wins":
                last["driver_wins"],

            "driver_standings_pos":
                last["driver_standings_pos"],


            "driver_avg_finish_last5":
                last["driver_avg_finish_last5"],

            "driver_best_finish_last5":
                last["driver_best_finish_last5"],

            "driver_top3_rate_last5":
                last["driver_top3_rate_last5"],

            "driver_circuit_avg_finish":
                last["driver_circuit_avg_finish"],


            "constructor_points":
                last["constructor_points"],

            "constructor_wins":
                last["constructor_wins"],

            "constructor_standings_pos":
                last["constructor_standings_pos"],


            "constructor_avg_finish_last5":
                last["constructor_avg_finish_last5"],


            "driver_enc":
                driver_encoder.transform(
                    [str(driver)]
                )[0],


            "constructor_enc":
                constructor_encoder.transform(
                    [str(constructor)]
                )[0],


            "circuit_enc":
                circuit_encoder.transform(
                    [circuit]
                )[0],


            # Use historical weather
            # until forecast data available

            "weather_warm":
                last["weather_warm"],

            "weather_cold":
                last["weather_cold"],

            "weather_dry":
                last["weather_dry"],

            "weather_wet":
                last["weather_wet"],

            "weather_cloudy":
                last["weather_cloudy"]

        })


future_2026 = pd.DataFrame(
    future_rows
)


# ------------------------------------------------------------
# Predict
# ------------------------------------------------------------
print("future races:")
print(future_races)

print(
    "future rows created:",
    len(future_rows)
)

print(
    "future dataframe:",
    future_2026.shape
)

print(
    future_2026.columns.tolist()
)



future_2026["top3_prob"] = (
    final_model
    .predict_proba(
        future_2026[features]
    )[:,1]
)



# ------------------------------------------------------------
# Select predicted podium
# ------------------------------------------------------------

predicted_podium_2026 = (

    future_2026
    .sort_values(
        [
            "round",
            "top3_prob"
        ],
        ascending=[
            True,
            False
        ]
    )
    .groupby("round")
    .head(3)

)


# ------------------------------------------------------------
# Save CSV
# ------------------------------------------------------------

output_file = (
    "histgradientWQRolling2026.csv"
)


predicted_podium_2026.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)


print(
    "Saved:",
    output_file
)


print(
    predicted_podium_2026[
        [
            "round",
            "driver",
            "constructor",
            "top3_prob"
        ]
    ]
)


(2570, 41)
   season  round     circuit_id  \
0    2020      1  red_bull_ring   
1    2020      2  red_bull_ring   
2    2020      3    hungaroring   
3    2020      4    silverstone   
4    2020      5    silverstone   

                                                 url  \
0  https://en.wikipedia.org/wiki/2020_Austrian_Gr...   
1  https://en.wikipedia.org/wiki/2020_Styrian_Gra...   
2  https://en.wikipedia.org/wiki/2020_Hungarian_G...   
3  https://en.wikipedia.org/wiki/2020_British_Gra...   
4  https://en.wikipedia.org/wiki/70th_Anniversary...   

                       weather  weather_warm  weather_cold  weather_dry  \
0                        Sunny             1             0            0   
1                        Sunny             1             0            0   
2  Wet at start, partly cloudy             0             0            0   
3                Partly cloudy             0             0            0   
4                        Sunny             1             0        